# Notebook 3: a vaccine network from Pune

This notebook replicates Section 5.3 of *Topology Persistence under Target Insertion in Ramified Optimal Transport*, the application of the paper: the doses of the Covishield vaccine made by the Serum Institute of India in Pune and shipped to the seven neighboring countries of India in the first quarter of 2021. It builds the instance from the data files, chooses the map projection, solves every subset of the recipients exactly, follows the order of the first despatches with the sequential, joint, and windowed insertions, repeats the step for all 840 orders, evaluates the gap certificate with rigorous lower bounds, maps the demand that the network can absorb without changing its topology, compares the network with one sized by population, and draws the maps. It rebuilds Table 3 and the vaccine rows of Tables 4 and 5, compares every table with the file in `overleaf_A/tables`, and checks every number of the study against the results stored in `figures/vaccine_pune` (written by `code/vaccine_pune.py`). Notebook 1 covers the theory of Section 2 and the exact solver, notebook 2 the ladders, the certificate, and the batch insertion on the synthetic instances.

**How to run.** Open the notebook in Google Colab and choose *Runtime > Run all*; there is nothing to upload, mount, or find. The first code cell unpacks the code and data embedded in the notebook (63 files, 17 MB) into the folder `/content/ArticleA` and makes it the working directory: the library in `colab/` (`rot_core.py`, `rot_tools.py`), the data in `data/vaccine_pune` (the three raw files with their `SOURCES.md` and, in `processed/`, the instance in tidy form) and `data/maps`, with the data dictionary `data/DATA_DICTIONARY.md`, the stored results and reference maps in `figures/vaccine_pune`, the stored results of the synthetic ladders in `figures/inductive` that Tables 4 and 5 share with notebook 2, the tables in `overleaf_A/tables`, and four files of `code/`. Three of them draw the maps of Section 12: `figure_vaccine_maps.py`, with `vaccine_pune.py` and `plotstyle.py`, which it imports. The fourth, `make_vaccine_data.py`, wrote the files of `processed/` (Section 2). Every path named below is inside that folder. Outside Colab, in Jupyter or VS Code, the same cell writes the folder `ArticleA_runtime` next to the notebook. No GPU and no paid service are needed.

**Default run and full run.** By default every long loop runs a documented subset, compares each recomputed number with the stored one, and prints `PASS` or `FAIL`; the tables, the maps, and the numbers of the text come from the stored results. The loops run in parallel processes, at most six (`RT.MAX_WORKERS`, or the environment variable `ROT_MAX_WORKERS`). The default run took about four minutes on the laptop used for the paper with six processes; a free Colab session, with two slower cores, takes about three times as long. Setting `RUN_FULL = True` reruns every study of `code/vaccine_pune.py` (hours on a laptop, days on Colab):

| Section | Default subset | `RUN_FULL = True` |
|---|---|---|
| 3, exact trees | the 210 subsets of three and four targets and the five-target prefix of the date order, three exponents | all 300 solves: 99 subsets per exponent and the population variant (a seven-target solve takes about 8 minutes on one core) |
| 4, date order | the four date-consistent orders at three exponents and the order of the map | the same |
| 5, batch insertion | $\omega=0$ from three to seven targets; $\omega=1,2$ from three to five | all nine runs to seven targets |
| 6, order study | seven orders: the largest misses of the sequential induction and of the joint insertion at each exponent, and the order of the map | all $35\times24=840$ orders per exponent |
| 7, extension step | the 420 rungs from three targets | all 882 rungs |
| 8, certificate | the rungs $3\to4$ and $4\to5$ of the date order, three exponents | all 12 rungs |
| 9, full polish | 210 of the 10 395 seven-target topologies per exponent, the optimal ones, and the one with the least lower bound; 35 of the population variant | all 10 395 per exponent, with dose and with population masses |
| 10, threshold map | the whole map at the three exponents | the same |
| 11, population variant | the cross re-optimizations | the same, with the exact population trees |

## 0. Setup

In [1]:
# @title Unpack the code and data embedded in this notebook
# Nothing has to be uploaded, mounted, or found. PAYLOAD, at the end of this cell, is a zip archive in base64 of the
# files this notebook reads and of the two files that its text names (code/make_vaccine_data.py and
# data/DATA_DICTIONARY.md), with their paths in the folder ArticleA of the paper. The cell writes them to a working
# folder, /content/ArticleA in Google Colab and ArticleA_runtime next to the notebook elsewhere, puts that folder and
# its subfolders colab/ and code/ on sys.path, makes it the working directory, and lists what it wrote.
import base64, importlib.util, io, os, sys, zipfile


def running_in_colab():
    """True in Google Colab, where the folder /content exists and the module google.colab is available."""
    if not os.path.isdir("/content"):
        return False
    if "google.colab" in sys.modules:
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except (ImportError, ValueError):
        return False


def same_content(path, data):
    """True when the file at path exists and holds exactly data."""
    if not os.path.isfile(path) or os.path.getsize(path) != len(data):
        return False
    with open(path, "rb") as f:
        return f.read() == data


def unpack(payload):
    """Extract the embedded archive into the working folder and enter it; returns the folder and whether it is Colab."""
    in_colab = running_in_colab()
    if in_colab:
        root = "/content/ArticleA"
    else:
        here = os.getcwd()                  # Jupyter and VS Code start the kernel in the folder of the notebook
        root = here if os.path.basename(here) == "ArticleA_runtime" else os.path.join(here, "ArticleA_runtime")
    listing = []
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z:
        for info in z.infolist():
            parts = info.filename.split("/")
            if info.is_dir() or info.filename.startswith("/") or ".." in parts:
                continue
            data = z.read(info)             # checks the CRC of the entry
            target = os.path.join(root, *parts)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            if not same_content(target, data):
                with open(target + ".part", "wb") as f:      # a whole file or none, also when two notebooks
                    f.write(data)                            # unpack into the same folder at the same time
                os.replace(target + ".part", target)
            listing.append((info.filename, len(data)))
    for sub in ("", "code", "colab"):       # in the end colab/ comes first on sys.path, then code/, then the folder
        folder = os.path.join(root, sub) if sub else root
        if os.path.isdir(folder) and folder not in sys.path:
            sys.path.insert(0, folder)
    os.chdir(root)
    where = root if in_colab else "ArticleA_runtime, next to this notebook"
    print(f"unpacked {len(listing)} files, {sum(n for _, n in listing) / 1e6:.1f} MB, into {where}:")
    for folder in sorted({os.path.dirname(name) for name, _ in listing}):
        names = [os.path.basename(name) for name, _ in listing if os.path.dirname(name) == folder]
        print(f"  {folder}/: {', '.join(names)}")
    return root, in_colab

PAYLOAD = """
...
"""
ROOT, IN_COLAB = unpack(PAYLOAD)

unpacked 63 files, 16.9 MB, into /content/ArticleA:
  code/: figure_vaccine_maps.py, make_vaccine_data.py, plotstyle.py, vaccine_pune.py
  colab/: rot_core.py, rot_tools.py
  data/: DATA_DICTIONARY.md
  data/maps/: borders.geojson, coastline.geojson, countries.geojson, lakes.geojson
  data/vaccine_pune/: SOURCES.md, mea_supplies_neighbors.csv, population_2021.csv, sites.csv
  data/vaccine_pune/processed/: distances.csv, insertion_orders.csv, instance_meta.json, nodes_projected.csv, projection_candidates.csv, targets.csv, tranches_q1_2021.csv
  figures/inductive/: certificate_ladder.json, inductive_ladder.json, inductive_ladder_hard.json
  figures/inductive/cement-a050/: summary.csv
  figures/inductive/cement-a050-w1/: summary.csv
  figures/inductive/cement-a075/: summary.csv
  figures/inductive/cement-a075-w1/: summary.csv
  figures/inductive/cement-a085/: summary.csv
  figures/inductive/cement-a085-aer/: summary.csv
  figures/inductive/cement-a085-w1/: summary.csv
  figures/inductive/

In [2]:
# Setup: the same cell works in Google Colab and locally. ROOT is the working folder written by the cell above.
import os, sys, importlib.util, subprocess
for var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "1")            # one BLAS thread per process; the long loops use processes instead

if IN_COLAB:
    # Latin Modern gives the figures the face of the manuscript; without it they use matplotlib's Computer Modern
    subprocess.run("apt-get -qq install fonts-lmodern > /dev/null 2>&1", shell=True)

# pip installs only what is missing (Colab already has all of these)
missing = [p for p in ("numpy", "scipy", "matplotlib", "pandas") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

COLAB_DIR = os.path.join(ROOT, "colab")       # rot_core.py, rot_tools.py, plotstyle.py
FIG = os.path.join(ROOT, "figures")            # stored results of the paper
TABLES = os.path.join(ROOT, "overleaf_A", "tables")
OUT = os.path.join(COLAB_DIR, "output")        # everything this notebook writes
for sub in ("figures", "tables", "results"):
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
print("folder", os.path.basename(ROOT), "| Colab" if IN_COLAB else "| local", "| cores:", os.cpu_count())

folder ArticleA | Colab | cores: 2


In [3]:
RUN_FULL = False      # True reruns every study in full (see the table above)

import filecmp, glob, itertools, json, time
import numpy as np
import pandas as pd
from IPython.display import Image, display
import rot_core as RC       # geometry of a fixed topology, enumeration of full topologies, exact solver
import rot_tools as RT      # data, projection, and study steps of the vaccine network; insertion steps; tables
ck = RT.Checker()           # collects the PASS/FAIL lines of this notebook
T_START = time.perf_counter()
VFIG = os.path.join(FIG, "vaccine_pune")           # stored results of the vaccine network
DATA = os.path.join(ROOT, "data", "vaccine_pune")   # the three data files and their sources

def load(name):
    """A stored result of the vaccine study (figures/vaccine_pune)."""
    return json.load(open(os.path.join(VFIG, name)))

def strip(x, drop=("P", "time_s")):
    """A result without the positions of its branching points: costs, gaps, topologies (edges), and flows are
    compared with the stored ones, and the positions follow from them up to the tolerance of the solver."""
    if isinstance(x, dict):
        return {k: strip(v, drop) for k, v in x.items() if k not in drop}
    if isinstance(x, (list, tuple)):
        return [strip(v, drop) for v in x]
    return x

def tex_num(x, d):
    """A float printed with d decimals, as the tables of the paper print it."""
    return f"{x:.{d}f}"

INST = load("instance.json")          # masses, orders, projection, coordinates (vaccine_pune.py instance)
EXACT = RT.load_vaccine_exact()       # exact trees with ties of every subset (vaccine_pune.py exact)
NAMES = RT.primary_order()            # the date order of the first despatches
print(len(EXACT), "stored exact solves; date order:", ", ".join(NAMES))

300 stored exact solves; date order: BTN, MDV, BGD, NPL, MMR, LKA, AFG


## 1. Data and sources

Three small files in `data/vaccine_pune` define the instance, and `SOURCES.md` in the same folder gives the provenance of every number.

- `mea_supplies_neighbors.csv`: every tranche to the neighbors of India in the table *Vaccine Supply* of the Ministry of External Affairs (MEA), as on 15 June 2023: the quantity in lakh doses (1 lakh = 100 000), the channel (grant, commercial, or COVAX), the date of despatch, and the producer label of the table, SII for the Serum Institute of India in Pune (Covishield) and BB for Bharat Biotech in Hyderabad (Covaxin).
- `sites.csv`: the source $O$ at Pune airport (PNQ), about 10 km from the plant of the Serum Institute, and the main international airport of each recipient, with coordinates from the Natural Earth airports layer; Paro (PBH), which the layer lacks, has its published coordinates.
- `population_2021.csv`: the 2021 populations of the World Bank, World Development Indicators (indicator SP.POP.TOTL), used by the population variant of Section 11.

The maps of Section 12 draw the Natural Earth 1:50m layers of countries and lakes (public domain), cut to the region and stored in `data/maps`.

In [4]:
print(open(os.path.join(DATA, "SOURCES.md"), encoding="utf-8").read())
supplies = pd.read_csv(os.path.join(DATA, "mea_supplies_neighbors.csv"), keep_default_na=False)
sites = pd.read_csv(os.path.join(DATA, "sites.csv"))
pop = pd.read_csv(os.path.join(DATA, "population_2021.csv"))
print(f"{len(supplies)} tranches to {supplies.country.nunique()} countries, "
      f"{supplies.quantity_lakh_doses.sum():.3f} lakh doses in all")
display(supplies)
display(sites)
display(pop)

# Sources of the Pune vaccine instance (accessed 2026-10-11)

## Supplies: `mea_supplies_neighbors.csv`

Ministry of External Affairs, Government of India, "Vaccine Supply", https://www.mea.gov.in/vaccine-supply.htm, table "Supplies so far (In lakhs) (As on June 15, 2023 at 1000 hrs)", columns Grant, Commercial, and COVAX, each with Quantity and Date of Despatch.

- Units: lakh doses (1 lakh = 100 000 doses).
- `producer_label`: SII (Serum Institute of India, Pune; Covishield) or BB (Bharat Biotech, Hyderabad; Covaxin), as labeled in the table. The footnote of the table reads "SII: Serum Institute of Technology" and "BB: Bharat Biotech 's Covaxin". A further footnote reads "Wherever not mentioned, Covidshield has been supplied. Covaxin and Covovax are mentioned separately." Covishield is made by the Serum Institute in Pune, so unlabeled tranches are taken to be SII doses; in the first quarter of 2021 the only tranches labeled BB were sent to Myanmar (2 lakh, 11 February) and Mauritius 

,country,iso3,channel,quantity_lakh_doses,date_of_despatch,producer_label
0,Bhutan,BTN,grant,1.500,2021-01-20,
1,Bhutan,BTN,grant,4.000,2021-03-21,
2,Maldives,MDV,grant,1.000,2021-01-20,
3,Maldives,MDV,grant,1.000,2021-02-19,
4,Maldives,MDV,commercial,1.000,2021-03-29,
5,Maldives,MDV,covax,0.120,2021-03-06,
6,Bangladesh,BGD,grant,20.000,2021-01-21,
7,Bangladesh,BGD,grant,12.000,2021-03-26,
8,Bangladesh,BGD,grant,1.000,2021-04-02,
9,Bangladesh,BGD,commercial,50.000,2021-01-25,


,node,role,country,iso3,iata,name,lat,lon,coordinate_source
0,O,source,India,IND,PNQ,Pune,18.579177,73.908984,Natural Earth 10m airports
1,BTN,target,Bhutan,BTN,PBH,Paro International Airport,27.408890,89.420560,Wikipedia (Paro International Airport)
2,MDV,target,Maldives,MDV,MLE,Male International (Velana),4.188701,73.527390,Natural Earth 10m airports
3,BGD,target,Bangladesh,BGD,DAC,Dhaka (Hazrat Shahjalal),23.848124,90.404924,Natural Earth 10m airports
4,NPL,target,Nepal,NPL,KTM,Tribhuvan International,27.700282,85.357140,Natural Earth 10m airports
5,MMR,target,Myanmar,MMR,RGN,Yangon (Mingaladon),16.901154,96.134195,Natural Earth 10m airports
6,LKA,target,Sri Lanka,LKA,CMB,Colombo (Katunayake),7.178077,79.885257,Natural Earth 10m airports
7,AFG,target,Afghanistan,AFG,KBL,Kabul International,34.563398,69.210074,Natural Earth 10m airports


,iso3,country,year,population,source
0,AFG,Afghanistan,2021,40000412,World Bank WDI SP.POP.TOTL
1,BGD,Bangladesh,2021,167658854,World Bank WDI SP.POP.TOTL
2,BTN,Bhutan,2021,775442,World Bank WDI SP.POP.TOTL
3,LKA,Sri Lanka,2021,22156000,World Bank WDI SP.POP.TOTL
4,MDV,Maldives,2021,516154,World Bank WDI SP.POP.TOTL
5,MMR,Myanmar,2021,53387102,World Bank WDI SP.POP.TOTL
6,NPL,Nepal,2021,29475010,World Bank WDI SP.POP.TOTL


### Selection of the first-quarter doses

The masses are the doses produced in Pune and dispatched in the first quarter of the program. Let $\mathcal Q_i$ be the tranches to recipient $i$ that are not labeled BB and left between 20 January and 31 March 2021, and let $q$ be the quantity of a tranche in lakh doses. The mass of recipient $i$, in millions of doses, is

$$m_i=\frac{1}{10}\sum_{q\in\mathcal Q_i}q .$$

Unlabeled tranches count as doses of the Serum Institute. In this window the only tranche to one of the seven recipients labeled BB is the grant of 2 lakh doses of Covaxin to Myanmar on 11 February 2021. The order of insertion is the date of the first despatch, and two recipients served on the same day are ordered by the larger first-day tranche.

In [5]:
sel = RT.vaccine_tranches()                              # the tranches of the instance
print(pd.DataFrame([{**r, "date": r["date"].isoformat()} for r in sel]).to_string(index=False))
masses = RT.vaccine_masses()                             # million doses
first = RT.first_despatch()
total = sum(masses.values())
print("\nmasses (million doses):", masses, f"total {total:.3f}")
for t in NAMES:
    print(f"  {t}: first despatch {first[t][0]}, {first[t][1]:g} lakh doses that day")
orders = RT.date_consistent_orders()
print("date order:", ", ".join(NAMES))
print("date-consistent orders:", [", ".join(o) for o in orders])

in_window = supplies[(supplies.date_of_despatch >= "2021-01-20") & (supplies.date_of_despatch <= "2021-03-31")
                     & supplies.iso3.isin(RT.VACCINE_TARGETS)]
bb = in_window[in_window.producer_label == "BB"]
ck("masses from the CSV: BTN 0.55, MDV 0.312, BGD 10.2, NPL 2.448, MMR 3.5, LKA 1.264, AFG 0.968; total 19.242",
   masses == {"BTN": 0.55, "MDV": 0.312, "BGD": 10.2, "NPL": 2.448, "MMR": 3.5, "LKA": 1.264, "AFG": 0.968}
   and round(total, 3) == 19.242 and masses == INST["masses_million_doses"])
ck("the only BB tranche in the window is Myanmar, 2 lakh on 11 February 2021",
   len(bb) == 1 and bb.iso3.iloc[0] == "MMR" and bb.quantity_lakh_doses.iloc[0] == 2 and bb.date_of_despatch.iloc[0] == "2021-02-11")
ck("date order BTN, MDV, BGD, NPL, MMR, LKA, AFG (ties broken by 1.5 > 1 and 20 > 10 lakh); four date-consistent orders",
   NAMES == ("BTN", "MDV", "BGD", "NPL", "MMR", "LKA", "AFG") and first["BTN"][1] == 1.5 and first["MDV"][1] == 1.0
   and first["BGD"][1] == 20 and first["NPL"][1] == 10 and len(orders) == 4
   and [list(o) for o in orders] == INST["date_consistent_orders"])

iso3    channel  lakh       date
 BTN      grant  1.50 2021-01-20
 BTN      grant  4.00 2021-03-21
 MDV      grant  1.00 2021-01-20
 MDV      grant  1.00 2021-02-19
 MDV commercial  1.00 2021-03-29
 MDV      covax  0.12 2021-03-06
 BGD      grant 20.00 2021-01-21
 BGD      grant 12.00 2021-03-26
 BGD commercial 50.00 2021-01-25
 BGD commercial 20.00 2021-02-22
 NPL      grant 10.00 2021-01-21
 NPL      grant  1.00 2021-03-28
 NPL commercial 10.00 2021-02-20
 NPL      covax  3.48 2021-03-05
 MMR      grant 15.00 2021-01-22
 MMR commercial 20.00 2021-02-11
 LKA      grant  5.00 2021-01-28
 LKA commercial  5.00 2021-02-24
 LKA      covax  2.64 2021-03-06
 AFG      grant  5.00 2021-02-07
 AFG      covax  4.68 2021-03-06

masses (million doses): {'BTN': 0.55, 'MDV': 0.312, 'BGD': 10.2, 'NPL': 2.448, 'MMR': 3.5, 'LKA': 1.264, 'AFG': 0.968} total 19.242
  BTN: first despatch 2021-01-20, 1.5 lakh doses that day
  MDV: first despatch 2021-01-20, 1 lakh doses that day
  BGD: first despatch 2021-

## 2. The projection and its error

Air freight follows great circles, while the model measures straight segments in a plane, so latitude $\varphi$ and longitude $\lambda$ are projected once to kilometers for the whole study. Great-circle distances use the haversine formula on the sphere of radius $R=6371$ km,

$$\delta(p,q)=2R\arcsin\sqrt{\sin^{2}\frac{\varphi_q-\varphi_p}{2}+\cos\varphi_p\cos\varphi_q\sin^{2}\frac{\lambda_q-\lambda_p}{2}} .$$

Three projections are compared, all centered at the midpoint $(\varphi_0,\lambda_0)$ of the latitude and longitude ranges of the eight sites; the two conics have their standard parallels one sixth of the latitude range inside its ends. The azimuthal equidistant projection sends a point at angular distance $c$ from the center,

$$\cos c=\sin\varphi_0\sin\varphi+\cos\varphi_0\cos\varphi\cos(\lambda-\lambda_0),$$

to

$$x=R\,\frac{c}{\sin c}\,\cos\varphi\sin(\lambda-\lambda_0),\qquad y=R\,\frac{c}{\sin c}\bigl(\cos\varphi_0\sin\varphi-\sin\varphi_0\cos\varphi\cos(\lambda-\lambda_0)\bigr),$$

which keeps every distance from the center. The Lambert conformal conic and the equidistant conic follow the spherical formulas in `RT.lcc` and `RT.eqdc`. The projection of the study, $X$, is the candidate with the smallest maximum relative error over the 28 pairs of sites,

$$e=\max_{p\neq q}\left|\frac{\|X(p)-X(q)\|}{\delta(p,q)}-1\right| .$$

The straight segment between two projected sites is then close to the image of the great circle that a flight follows; `RT.great_circle_departure` (`vaccine_pune.great_circle_departure`) samples the projected great-circle arc of each pair of sites at 401 points and measures its largest distance from the straight segment.

In [6]:
err = RT.projection_errors()
name, fun, par, e_max = RT.chosen_projection()
print(pd.DataFrame({k: {"max error (%)": 100 * v["max_abs_rel_error"], "mean error (%)": 100 * v["mean_abs_rel_error"],
                        "worst pair": "-".join(v["worst_pair"])} for k, v in err.items()}).T.to_string())
print(f"\nchosen: {name}, center {par['lat0']:.4f} N {par['lon0']:.4f} E, maximum error {100 * e_max:.3f}%")
pairs = pd.DataFrame(err[name]["pairs"]).T.astype(float)
print(pairs.assign(rel_error_pct=100 * pairs.rel_error).drop(columns="rel_error").round(3).to_string())

s, T, m = RT.vaccine_instance()
print("\ncoordinates (km):", {k: np.round(v, 2).tolist() for k, v in zip(("O",) + NAMES, np.vstack([s, T]))})
ck("projection: azimuthal equidistant centered at 19.3760 N, 82.6721 E, max error 0.542% (MDV-AFG), mean 0.246%",
   name == "azimuthal equidistant" and round(par["lat0"], 4) == 19.376 and round(par["lon0"], 4) == 82.6721
   and round(100 * e_max, 3) == 0.542 and err[name]["worst_pair"] == ["MDV", "AFG"]
   and round(100 * err[name]["mean_abs_rel_error"], 3) == 0.246)
ck("the conics: Lambert 1.520%, equidistant 1.503%, worst pair O-MMR for both",
   round(100 * err["Lambert conformal conic"]["max_abs_rel_error"], 3) == 1.520
   and round(100 * err["equidistant conic"]["max_abs_rel_error"], 3) == 1.503
   and err["Lambert conformal conic"]["worst_pair"] == err["equidistant conic"]["worst_pair"] == ["O", "MMR"])
ck("coordinates and the population masses as stored in instance.json",
   np.allclose(np.vstack([s, T]), [INST["coordinates_km"][k] for k in ("O",) + NAMES], rtol=0, atol=1e-9)
   and all(RT.agrees({k: v for k, v in err[n].items() if k != "pairs"}, INST["projection"]["candidates"][n])[0] for n in err))
gcd = RT.great_circle_departure()
print(f"\nlargest departure of a projected great circle from the straight segment: {gcd['max_km']:.1f} km "
      f"({'-'.join(gcd['worst_pair'])}, a segment of {gcd['pairs']['-'.join(gcd['worst_pair'])]['segment_km']:.0f} km); "
      f"from Pune {gcd['max_from_source_km']:.1f} km ({'-'.join(gcd['worst_pair_from_source'])})")
ck("straight segments stay within 28 km of the projected great circles (27.9 km, MDV-AFG) and within 6.5 km on the "
   "routes from Pune, as stored in instance.json",
   round(gcd["max_km"], 1) == 27.9 and gcd["worst_pair"] == ["MDV", "AFG"] and round(gcd["max_from_source_km"], 1) == 6.5
   and RT.agrees(gcd, INST["projection"]["great_circle_departure"])[0])

                        max error (%) mean error (%) worst pair
azimuthal equidistant        0.542299       0.245755    MDV-AFG
Lambert conformal conic      1.519688       0.840975      O-MMR
equidistant conic            1.503416       0.419431      O-MMR

chosen: azimuthal equidistant, center 19.3760 N 82.6721 E, maximum error 0.542%
         great_circle_km  projected_km  rel_error_pct
O-BTN           1864.108      1865.510          0.075
O-MDV           1600.685      1606.248          0.348
O-BGD           1805.988      1806.403          0.023
O-NPL           1547.150      1549.028          0.121
O-MMR           2359.704      2359.860          0.007
O-LKA           1423.101      1427.470          0.307
O-AFG           1837.087      1843.527          0.351
BTN-MDV         3083.407      3083.453          0.001
BTN-BGD          408.046       409.415          0.335
BTN-NPL          401.870       403.332          0.364
BTN-MMR         1356.881      1363.212          0.467
BTN-LKA        

### The processed instance

`code/make_vaccine_data.py` applies the rules above to the three raw files and writes the instance in tidy form to `data/vaccine_pune/processed`; `data/DATA_DICTIONARY.md` describes every column. The folder holds the 21 tranches that make up the masses (`tranches_q1_2021.csv`), the seven targets in the date order with their first despatch, mass, population, and population-variant mass (`targets.csv`), the source and the targets in the plane of the study, in km (`nodes_projected.csv`), the 28 pairwise distances with their relative errors (`distances.csv`), the three candidate projections (`projection_candidates.csv`), the four orders that the dates allow (`insertion_orders.csv`), and the rule, the units, and the SHA-256 digests of the raw files (`instance_meta.json`). Floats are written in their shortest round-trip form, so reading the text gives back the computed double.

The cell below reads these files and checks that every number in them equals, bit for bit, the number this notebook has just computed from the raw files with `rot_tools.py`, a second implementation of the rules of `code/vaccine_pune.py`. A reader who wants the instance without the code can start from these files.

In [7]:
import csv, hashlib
PROC = os.path.join(DATA, "processed")

def read_rows(name):
    """Rows of a processed CSV as dictionaries of strings; float() of a field gives back the stored double."""
    with open(os.path.join(PROC, name), newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

p_tranches, p_targets, p_nodes = read_rows("tranches_q1_2021.csv"), read_rows("targets.csv"), read_rows("nodes_projected.csv")
p_dist, p_cand, p_orders = read_rows("distances.csv"), read_rows("projection_candidates.csv"), read_rows("insertion_orders.csv")
p_meta = json.load(open(os.path.join(PROC, "instance_meta.json"), encoding="utf-8"))
display(pd.DataFrame(p_targets))
display(pd.DataFrame(p_nodes).drop(columns=["name"]))

# the instance as read from the processed files: source s, targets T in the date order, masses m (million doses)
X_csv = np.array([[float(r["x_km"]), float(r["y_km"])] for r in p_nodes])
s_csv, T_csv = X_csv[0], X_csv[1:]
m_csv = np.array([float(r["mass_million_doses"]) for r in p_targets])
m_pop_csv = np.array([float(r["mass_population_variant_million_doses"]) for r in p_targets])
s_pop, T_pop, m_pop = RT.vaccine_instance("population")
orders_csv = {}
for r in p_orders:
    orders_csv.setdefault(int(r["order_id"]), []).append(r["iso3"])

ck("processed targets, tranches, and orders equal those built from the raw files (masses, population-variant "
   "masses, first despatches, the 21 tranches, the four date-consistent orders; bit for bit)",
   [r["iso3"] for r in p_targets] == list(NAMES) and np.array_equal(m_csv, m) and np.array_equal(m_pop_csv, m_pop)
   and all(r["first_despatch_date"] == first[r["iso3"]][0].isoformat()
           and float(r["first_day_lakh_doses"]) == first[r["iso3"]][1] for r in p_targets)
   and sorted((r["iso3"], float(r["quantity_lakh_doses"]), r["date_of_despatch"]) for r in p_tranches)
   == sorted((x["iso3"], x["lakh"], x["date"].isoformat()) for x in sel)
   and [tuple(orders_csv[k]) for k in sorted(orders_csv)] == [tuple(o) for o in orders])
pairs_chosen = err[name]["pairs"]
ck("processed coordinates (km), the 28 distances, and the three candidate projections equal those computed above "
   "(bit for bit)",
   [r["node"] for r in p_nodes] == ["O", *NAMES] and np.array_equal(X_csv, np.vstack([s, T]))
   and len(p_dist) == 28
   and all((float(r["great_circle_km"]), float(r["planar_km"]), float(r["rel_error"]))
           == tuple(pairs_chosen[f"{r['node_a']}-{r['node_b']}"][k] for k in ("great_circle_km", "projected_km", "rel_error"))
           for r in p_dist)
   and all(float(r["max_abs_rel_error"]) == err[r["projection"]]["max_abs_rel_error"]
           and float(r["mean_abs_rel_error"]) == err[r["projection"]]["mean_abs_rel_error"]
           and (r["chosen"] == "true") == (r["projection"] == name) for r in p_cand))
ck("instance_meta.json: the SHA-256 digests of the three raw files and the parameters of the projection",
   all(hashlib.sha256(open(os.path.join(ROOT, k), "rb").read()).hexdigest() == v["sha256"]
       for k, v in p_meta["raw_files"].items())
   and p_meta["projection"]["name"] == name and p_meta["projection"]["params"] == par
   and p_meta["primary_order"] == list(NAMES) and round(p_meta["total_million_doses"], 3) == 19.242)

,order,iso3,country,iata,first_despatch_date,first_day_lakh_doses,n_tranches,mass_million_doses,population_2021,mass_population_variant_million_doses
0,1,BTN,Bhutan,PBH,2021-01-20,1.5,2,0.55,775442,0.04752397911775831
1,2,MDV,Maldives,MLE,2021-01-20,1.0,4,0.312,516154,0.03163317426390036
2,3,BGD,Bangladesh,DAC,2021-01-21,20.0,4,10.2,167658854,10.275192569403371
3,4,NPL,Nepal,KTM,2021-01-21,10.0,4,2.448,29475010,1.806414612228532
4,5,MMR,Myanmar,RGN,2021-01-22,15.0,2,3.5,53387102,3.2718985051178975
5,6,LKA,Sri Lanka,CMB,2021-01-28,5.0,3,1.264,22156000,1.35785949346702
6,7,AFG,Afghanistan,KBL,2021-02-07,5.0,2,0.968,40000412,2.4514776664015216


,node,role,order,iso3,iata,lat,lon,x_km,y_km
0,O,source,0,IND,PNQ,18.579177,73.908984,-923.2831385356583,-65.44724466454932
1,BTN,target,1,BTN,PBH,27.40889,89.42056,668.0965141782538,908.0191092619083
2,MDV,target,2,MDV,MLE,4.188701,73.52739,-1025.9165043826004,-1668.412665728804
3,BGD,target,3,BGD,DAC,23.848124,90.404924,786.9070087391816,516.2226250595927
4,NPL,target,4,NPL,KTM,27.700282,85.35714,265.258183571633,927.9592449496382
5,MMR,target,5,MMR,RGN,16.901154,96.134195,1431.4154310313347,-221.44636125874115
6,LKA,target,6,LKA,CMB,7.178077,79.885257,-309.7862098209452,-1354.3586615083175
7,AFG,target,7,AFG,KBL,34.563398,69.210074,-1244.8333557656654,1749.8201759532515


PASS  processed targets, tranches, and orders equal those built from the raw files (masses, population-variant masses, first despatches, the 21 tranches, the four date-consistent orders; bit for bit)
FAIL  processed coordinates (km), the 28 distances, and the three candidate projections equal those computed above (bit for bit)
PASS  instance_meta.json: the SHA-256 digests of the three raw files and the parameters of the projection


## 3. Exact trees with ties

For a set $S$ of targets with masses $m_i$, the source carries $m(S)=\sum_{i\in S}m_i$, a full topology $T$ on $\lbrace O\rbrace\cup S$ has the cost $c(T)$ of its optimal realization,

$$c(T)=\min_{P}\sum_{e\in E(T)}|e|\,w(e)^{\alpha},$$

with $w(e)$ the mass that crosses the edge $e$, and

$$d(S)=\min_T c(T),\qquad \mathcal T^{\ast}(S)=\lbrace T:\ c(T)\le d(S)(1+10^{-7})\rbrace,\qquad g(S)=\min_{T\notin\mathcal T^{\ast}(S)}c(T)-d(S),$$

with $g(S)=+\infty$ when every topology is optimal. Costs are in km (million doses)$^{\alpha}$. The exact solver of Section 4.1 (`RC.solve_exhaustive`) scans the $(2k-5)!!$ full topologies on $k=|S|+1$ terminals, polishes the candidates, and returns every optimal topology; $g(S)$ comes from the polished costs (Section 9 replaces it by rigorous lower bounds for the seven-target instance). With seven targets there are 99 subsets of at least three, solved at $\alpha\in\lbrace0.5,0.75,0.85\rbrace$. `RT.vaccine_solve_subset` is `vaccine_pune.solve_subset`; the stored file also holds the population variant of Section 11.

In [8]:
tasks = RT.vaccine_exact_tasks() if RUN_FULL else (
    [("doses", a, sub) for a in RT.ALPHAS for n in (3, 4) for sub in itertools.combinations(range(7), n)]
    + [("doses", a, (0, 1, 2, 3, 4)) for a in RT.ALPHAS])
t0 = time.perf_counter()
fresh = RT.pmap(RT.vaccine_solve_subset, tasks)
print(f"{len(fresh)} exact solves in {time.perf_counter() - t0:.0f} s")

def same_exact(new, old):
    """Cost, gap, number of optimal topologies and of the trees they realize, and the optimal topologies."""
    keys = ("d", "g", "g_rel", "n_optimal", "n_trees", "n_polished", "branching_nondegenerate")
    ok, msg = RT.agrees({k: new[k] for k in keys}, {k: old[k] for k in keys}, rel=1e-7, atol=1e-9)
    tops = lambda r: {RT.split_of(t["edges"], r["n"] + 1) for t in r["ties"]}
    return ok and tops(new) == tops(old), msg

bad = [RT.vaccine_key(r["variant"], r["alpha"], tuple(r["subset"])) for r in fresh
       if not same_exact(r, EXACT[RT.vaccine_key(r["variant"], r["alpha"], tuple(r["subset"]))])[0]]
ck(f"exact trees: {len(fresh)} recomputed subsets agree with the stored ones (cost, gap, optimal topologies)",
   not bad, f"differences in {bad[:4]}" if bad else "")
if RUN_FULL:
    json.dump({RT.vaccine_key(r["variant"], r["alpha"], tuple(r["subset"])): r for r in fresh},
              open(os.path.join(OUT, "results", "exact_subsets.json"), "w"))

213 exact solves in 352 s
PASS  exact trees: 213 recomputed subsets agree with the stored ones (cost, gap, optimal topologies)


Along the date order the exact trees are written below as nested lists rooted at $O$: $(A,B)$ is a branching point with the branches $A$ and $B$, and $X(Y)$ means that $Y$ hangs from a branching point that sits on the recipient $X$, a degenerate branching point. The table gives $d_n$, the gap $g_n$ in percent of $d_n$, the number of optimal full topologies, and the number of nondegenerate branching points.

In [9]:
rows = []
for a in RT.ALPHAS:
    for n in range(3, 8):
        r = EXACT[RT.vaccine_key("doses", a, tuple(range(n)))]
        trees = sorted({RT.nested_tree(t["edges"], t["P"], ["O"] + list(NAMES[:n])) for t in r["ties"]})
        rows.append({"alpha": a, "n": n, "d_n": round(r["d"], 4), "g_n (%)": round(100 * r["g_rel"], 4),
                     "optimal topologies": r["n_optimal"], "branching points": r["branching_nondegenerate"],
                     "tree": " | ".join(trees)})
TREES = pd.DataFrame(rows)
print(TREES.to_string(index=False))

expected = {(0.5, 7): "O((((BGD(MMR),BTN),NPL),AFG),(LKA,MDV))", (0.75, 7): "O((((BGD,MMR),BTN),NPL),(LKA,MDV),AFG)",
            (0.85, 7): "O((((BGD,BTN),MMR),NPL),(LKA,MDV),AFG)", (0.5, 3): "O((BGD,BTN),MDV)",
            (0.5, 5): "O(((BGD(MMR),BTN),NPL),MDV)", (0.85, 5): "O((((BGD,BTN),MMR),NPL),MDV)"}
got = {(r["alpha"], r["n"]): r["tree"] for r in rows}
ck("trees of the date order: at alpha = 0.5 Myanmar hangs from Dhaka; at 0.75 and 0.85 three branches leave Pune",
   all(got[k] == v for k, v in expected.items()))
d7 = {a: EXACT[RT.vaccine_key("doses", a, tuple(range(7)))] for a in RT.ALPHAS}
ck("seven targets: d_7 = 14068.4596, 22253.0077, 27071.7230; g_7 = 0.2105, 0.0052, 0.0570%",
   [round(d7[a]["d"], 4) for a in RT.ALPHAS] == [14068.4596, 22253.0077, 27071.7230]
   and [round(100 * d7[a]["g_rel"], 4) for a in RT.ALPHAS] == [0.2105, 0.0052, 0.0570])
ck("seven targets at 0.75 and 0.85: three tied full topologies realizing a single tree",
   all(d7[a]["n_optimal"] == 3 and d7[a]["n_trees"] == 1 for a in (0.75, 0.85)) and d7[0.5]["n_optimal"] == 1)
doses = [r for r in EXACT.values() if r["variant"] == "doses"]
inf_gap = {a: [r["targets"] for r in doses if r["alpha"] == a and not np.isfinite(r["g"])] for a in RT.ALPHAS}
print("subsets with an infinite gap:", {a: len(v) for a, v in inf_gap.items()})
ck("99 subsets per alpha and 3 population solves; infinite gaps only on three-target subsets with AFG (0, 5, 8)",
   [sum(r["alpha"] == a for r in doses) for a in RT.ALPHAS] == [99] * 3 and len(EXACT) == 300
   and [len(inf_gap[a]) for a in RT.ALPHAS] == [0, 5, 8]
   and all(len(t) == 3 and "AFG" in t for v in inf_gap.values() for t in v))
near = [r for r in doses if r["n_trees"] > 1]
spread = [max(t["cost"] for t in r["ties"]) / r["d"] - 1 for r in near]
print("subsets whose optimal topologies realize more than one tree:",
      [(r["alpha"], r["targets"], r["n_optimal"], r["n_trees"], f"{s_:.1e}") for r, s_ in zip(near, spread)])
ck("one near tie: MDV, MMR, LKA, AFG at 0.75 has two trees within 7.6e-8 of each other",
   len(near) == 1 and near[0]["alpha"] == 0.75 and near[0]["targets"] == ["MDV", "MMR", "LKA", "AFG"]
   and round(spread[0], 9) == 7.6e-8)

 alpha  n        d_n  g_n (%)  optimal topologies  branching points                                    tree
  0.50  3  7118.0557   9.3496                   1                 1                        O((BGD,BTN),MDV)
  0.50  4  8324.4986   0.3138                   1                 2                  O(((BGD,BTN),NPL),MDV)
  0.50  5 11021.3921   0.2017                   1                 2             O(((BGD(MMR),BTN),NPL),MDV)
  0.50  6 12310.9411   0.1806                   1                 3       O(((BGD(MMR),BTN),NPL),(LKA,MDV))
  0.50  7 14068.4596   0.2105                   1                 4 O((((BGD(MMR),BTN),NPL),AFG),(LKA,MDV))
  0.75  3 11637.7158   4.3665                   1                 1                        O((BGD,BTN),MDV)
  0.75  4 13893.5573   0.2039                   1                 2                  O(((BGD,BTN),NPL),MDV)
  0.75  5 18857.8041   0.0061                   1                 3            O((((BGD,MMR),BTN),NPL),MDV)
  0.75  6 20453.9051   0.005

## 4. The date order: sequential induction and extension step

Write $S_n$ for the first $n$ targets of an order, $d_n=d(S_n)$, and $g_n=g(S_n)$. The *sequential induction* starts from the optimal topologies of $S_3$ and inserts one target at a time: at each step every held topology is extended by the new target on each of its edges, the geometry of every extension is optimized, and the extensions that tie with the best are kept (`RT.insert_best_edge_multi`, tie-aware, as in Section 4.1). The *extension step* at the rung $n-1\to n$ does the same from the optimal topologies of $S_{n-1}$. Its excess over $d_n$ is zero exactly when the extension property holds, that is, when some optimal topology of $S_n$ restricts to an optimal topology of $S_{n-1}$ (Definition 3.1 and Theorem 3.3). `RT.vaccine_order_chain` (`vaccine_pune.order_chain`) computes both along an order, with the exact trees of Section 3.

The cell runs the four orders consistent with the dates at the three exponents, and the order BTN, MDV, BGD, MMR, NPL, LKA, AFG at $\alpha=0.5$, which inserts Myanmar before Nepal and is drawn in panel (b) of the network map.

In [10]:
ORDER_MAP = ("BTN", "MDV", "BGD", "MMR", "NPL", "LKA", "AFG")      # the order of panel (b) of the network map
tasks = [(a, o) for a in RT.ALPHAS for o in RT.date_consistent_orders()] + [(0.5, ORDER_MAP)]
t0 = time.perf_counter()
chains = RT.pmap(RT.vaccine_order_chain, tasks)
print(f"{len(chains)} orders in {time.perf_counter() - t0:.0f} s")
stored = load("primary_order.json")
bad = []
for r, old in zip(chains[:-1], stored):
    ok, msg = RT.agrees(strip(r), strip(old), rel=1e-7, atol=1e-9)
    if not ok:
        bad.append(f"{r['alpha']} {r['order']}: {msg}")
ck(f"date order: {len(chains) - 1} recomputed chains agree with primary_order.json", not bad, "; ".join(bad[:2]))

for r in chains:
    print(f"alpha = {r['alpha']:<4} {'-'.join(r['order'])}: d_3 = {r['exact'][3]['d']:.3f}; sequential excess (%) "
          + " ".join(f"{100 * v['excess_rel']:+.4f}" for v in r["sequential"].values())
          + "; extension step exact on " + ", ".join(k for k, v in r["extension_step"].items() if v["exact"]))
date = chains[:-1]
ck("no miss along the date-consistent orders: sequential induction and extension step exact at every step and alpha",
   max(abs(v["excess_rel"]) for r in date for v in list(r["sequential"].values()) + list(r["extension_step"].values())) < 1e-9
   and all(v["exact"] and v["extension_property"] for r in date for v in r["extension_step"].values()))
other = [r for r in date if r["order"][:3] == ["BTN", "MDV", "NPL"]]
ck("the other base {BTN, MDV, NPL}: d_3 = 3878.420, 4457.393, 4778.285",
   sorted(round(r["exact"][3]["d"], 3) for r in other) == [3878.42, 4457.393, 4778.285])
swap = chains[-1]
print(f"order of the map: sequential excess {100 * swap['sequential'][7]['excess_rel']:.3f}% at seven targets")
ck("order of the map (MMR before NPL), alpha = 0.5: the sequential induction ends 0.255% above the optimum",
   round(100 * swap["sequential"][7]["excess_rel"], 3) == 0.255)
if os.path.exists(os.path.join(VFIG, "network_figure.json")):
    nf = load("network_figure.json")
    ok, msg = RT.agrees(strip(swap), strip(nf["swap"]), rel=1e-7, atol=1e-9)
    ck("the trees of the network map as stored in network_figure.json", ok, msg)

13 orders in 301 s
FAIL  date order: 12 recomputed chains agree with primary_order.json  [0.85 ['BTN', 'MDV', 'BGD', 'NPL', 'MMR', 'LKA', 'AFG']: .trees.sequential7.edges[3][0]: 4.0 against stored 8.0; 0.85 ['BTN', 'MDV', 'NPL', 'BGD', 'MMR', 'LKA', 'AFG']: .trees.sequential7.edges[0][0]: 8.0 against stored 0.0]
alpha = 0.5  BTN-MDV-BGD-NPL-MMR-LKA-AFG: d_3 = 7118.056; sequential excess (%) -0.0000 +0.0000 +0.0000 +0.0000; extension step exact on 3->4, 4->5, 5->6, 6->7
alpha = 0.5  BTN-MDV-NPL-BGD-MMR-LKA-AFG: d_3 = 3878.420; sequential excess (%) +0.0000 +0.0000 +0.0000 +0.0000; extension step exact on 3->4, 4->5, 5->6, 6->7
alpha = 0.5  MDV-BTN-BGD-NPL-MMR-LKA-AFG: d_3 = 7118.056; sequential excess (%) -0.0000 +0.0000 +0.0000 +0.0000; extension step exact on 3->4, 4->5, 5->6, 6->7
alpha = 0.5  MDV-BTN-NPL-BGD-MMR-LKA-AFG: d_3 = 3878.420; sequential excess (%) +0.0000 +0.0000 +0.0000 +0.0000; extension step exact on 3->4, 4->5, 5->6, 6->7
alpha = 0.75 BTN-MDV-BGD-NPL-MMR-LKA-AFG: d_3 

## 5. Joint and windowed insertion (Section 4.2) and the vaccine rows of Table 4

The joint insertion adds $b$ targets at once. A choice map $\varphi$ sends each new target to an edge of the current tree, and its insertion places one new branching point per target on that edge, joined to the target and ordered along the edge as the projections of the targets (Definition 3.8). The joint step searches every choice map, the reach

$$R_b(T),\qquad |R_b(T)|=(2k-3)^{b},$$

which is the feasible set of the one-hot program of Section 4.2 (Proposition 3.9). A window of $\omega$ targets detaches the $\omega$ most recent targets and inserts them again with the new ones on the reduced tree; the better of the plain and the windowed proposal is kept. On the vaccine network the steps are $3\to5\to7$ along the date order with $b=2$, from the exact three-target tree, each column continuing from the tree that the same method produced at the previous step. `RT.batch_ladder` is `hqts_inductive.run` restricted to the sequential and joint steps; the exact references come from Section 3, as in `vaccine_pune.py`.

In [11]:
ref = {a: {n: EXACT[RT.vaccine_key("doses", a, tuple(range(n)))]["d"] for n in (5, 7)} for a in RT.ALPHAS}
specs = [("vaccine", a, w, None if (RUN_FULL or w == 0) else 5, ref[a]) for a in RT.ALPHAS for w in (0, 1, 2)]
t0 = time.perf_counter()
results = RT.pmap(RT.batch_job, specs)
print(f"{len(specs)} runs in {time.perf_counter() - t0:.0f} s")
bad, n_rows = [], 0
for (inst, alpha, w, _, _), rows in zip(specs, results):
    tag = f"vaccine-a{int(round(alpha * 100)):03d}" + (f"-w{w}" if w else "")
    st_all = pd.read_csv(os.path.join(VFIG, "batch", tag, "summary.csv"))
    for r in rows:
        st = st_all[(st_all.method == r["method"]) & (st_all.n_before == r["n_before"])].iloc[0]
        n_rows += 1
        ok = (abs(r["cost"] - st.cost) <= 1e-9 * st.cost and r["evaluations"] == st.evaluations
              and abs(r["gap_pct"] - st.gap_pct) <= 1e-6)
        if not ok:
            bad.append(f"{tag} {r['method']} n = {r['n_before']}")
        print(f"{tag:16s} {r['n_before']} -> {r['n_after']} {r['method']:>10}: cost {r['cost']:.5f}, "
              f"excess {r['gap_pct']:+.1e}%, {r['evaluations']} geometry optimizations")
ck(f"batch insertion: {n_rows} recomputed steps agree with the stored rows (cost and number of evaluations)",
   not bad, f"{bad}" if bad else "")

9 runs in 98 s
vaccine-a050     3 -> 5 sequential: cost 11021.39205, excess +0.0e+00%, 12 geometry optimizations
vaccine-a050     3 -> 5      joint: cost 11021.39205, excess +0.0e+00%, 25 geometry optimizations
vaccine-a050     5 -> 7 sequential: cost 14068.45955, excess +2.2e-14%, 20 geometry optimizations
vaccine-a050     5 -> 7      joint: cost 14068.45955, excess +2.2e-14%, 81 geometry optimizations
vaccine-a050-w1  3 -> 5 sequential: cost 11021.39205, excess +0.0e+00%, 12 geometry optimizations
vaccine-a050-w1  3 -> 5      joint: cost 11021.39205, excess +0.0e+00%, 52 geometry optimizations
vaccine-a050-w2  3 -> 5 sequential: cost 11021.39205, excess +0.0e+00%, 12 geometry optimizations
vaccine-a050-w2  3 -> 5      joint: cost 11021.39205, excess +0.0e+00%, 25 geometry optimizations
vaccine-a075     3 -> 5 sequential: cost 18857.80415, excess +0.0e+00%, 12 geometry optimizations
vaccine-a075     3 -> 5      joint: cost 18857.80415, excess +0.0e+00%, 25 geometry optimizations
vacci

In [12]:
text = RT.table_batch_article(FIG, os.path.join(OUT, "tables", "tab_batch.tex"))
ok, msg = RT.compare_table(text, os.path.join(TABLES, "tab_batch.tex"))
ck("Table 4 identical to overleaf_A/tables/tab_batch.tex (vaccine rows first)", ok, msg)
print(text)
allb = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(VFIG, "batch", "*", "summary.csv")))])
print(allb.groupby("method").gap_pct.apply(lambda x: x.abs().max()))
ck("every vaccine step reaches the optimum: sequential, joint with omega = 0, 1, 2, and the stored annealing and dry-run QAOA",
   allb.gap_pct.abs().max() < 1e-9 and set(allb.method) == {"sequential", "joint", "sa", "qaoa"}
   and all(f"{0:.3f}" == c for ln in text.splitlines() if ln.startswith("Vaccine")
           for c in [x.strip() for x in ln.rstrip("\\ ").split("&")[3:]] if c != "--"))

PASS  Table 4 identical to overleaf_A/tables/tab_batch.tex (vaccine rows first)  [identical]
\begin{tabular}{lrlrrrr}
\toprule
 & & & & \multicolumn{3}{c}{Joint, $\omega=$} \\
\cmidrule(lr){5-7}
Instance & $\alpha$ & Step & Sequential & 0 & 1 & 2 \\
\midrule
Vaccine network & 0.5 & $3\to5$ & 0.000 & 0.000 & 0.000 & -- \\
Vaccine network & 0.5 & $5\to7$ & 0.000 & 0.000 & 0.000 & 0.000 \\
Vaccine network & 0.75 & $3\to5$ & 0.000 & 0.000 & 0.000 & -- \\
Vaccine network & 0.75 & $5\to7$ & 0.000 & 0.000 & 0.000 & 0.000 \\
Vaccine network & 0.85 & $3\to5$ & 0.000 & 0.000 & 0.000 & -- \\
Vaccine network & 0.85 & $5\to7$ & 0.000 & 0.000 & 0.000 & 0.000 \\
Disk, unequal masses & 0.5 & $3\to5$ & 0.630 & 0.630 & 0.000 & -- \\
Disk, unequal masses & 0.5 & $5\to7$ & 1.034 & 1.034 & 1.092 & 0.000 \\
Two clusters & 0.75 & $3\to5$ & 0.000 & 0.000 & 0.000 & -- \\
Two clusters & 0.75 & $5\to7$ & 0.006 & 0.006 & 0.006 & 0.006 \\
\bottomrule
\end{tabular}

method
joint         3.774758e-13
qaoa          3

## 6. Every order of insertion and Table 3

The dates fix one order, and four orders are consistent with them. To see how much the result depends on the order, the study takes every base set of three targets and every order of the other four, $\binom{7}{3}\times4!=35\times24=840$ orders per exponent. For each order `RT.vaccine_order_rows` (`vaccine_pune.order_base`) computes

- the sequential induction from the exact base tree and its excess over $d_7$;
- the joint insertion in two-target steps $3\to5\to7$ without window, from every optimal base topology; the second step starts from every first step within the tie tolerance (`RT.step_joint_ties`), so the result does not depend on how ties among first steps are broken;
- the joint insertion of the last two targets from the exact five-target tree;
- whether some optimal seven-target topology restricts to an optimal base topology (the extension property with $b=4$), and the extension property on every rung of the order.

A joint step depends on the set of targets it inserts and not on their order, so the archive computes it once per set, in the first permutation that contains it, and shares it; `RT.vaccine_order_rows` uses the same permutation, so a subset of orders returns the numbers of the full run. A miss is an excess above the tie tolerance $10^{-7}$. The default run recomputes, at each exponent, the order with the largest miss of the sequential induction and the order with the largest miss of the joint insertion, and the order of the network map.

In [13]:
STUDY = load("order_study.json")                 # 105 base tasks: 35 base sets x 3 exponents, 24 orders each
rows_all = [x for r in STUDY for x in r["rows"]]
key = lambda x: (x["alpha"], tuple(x["base"]), tuple(x["order"]))
if RUN_FULL:
    tasks = [(a, b, None) for a in RT.ALPHAS for b in itertools.combinations(range(7), 3)]
else:
    pick = []
    for a in RT.ALPHAS:
        rs = [x for x in rows_all if x["alpha"] == a]
        pick += [max(rs, key=lambda x: x["sequential_gap_rel"]), max(rs, key=lambda x: x["joint_steps_gap_rel"])]
    pick += [x for x in rows_all if key(x) == (0.5, ORDER_MAP[:3], ORDER_MAP[3:])]
    tasks = [(x["alpha"], tuple(NAMES.index(t) for t in x["base"]), [[NAMES.index(t) for t in x["order"]]]) for x in pick]
t0 = time.perf_counter()
fresh_tasks = RT.pmap(RT.vaccine_order_rows, tasks)
fresh = [x for r in fresh_tasks for x in r["rows"]]
print(f"{len(fresh)} orders in {(time.perf_counter() - t0) / 60:.1f} min")
stored_by_key = {key(x): x for x in rows_all}
bad = [key(x) for x in fresh if not RT.agrees(x, stored_by_key[key(x)], rel=1e-7, atol=1e-9)[0]]
for x in fresh:
    print(f"alpha = {x['alpha']:<4} base {'-'.join(x['base'])}, then {'-'.join(x['order'])}: sequential "
          f"{100 * x['sequential_gap_rel']:.3f}%, joint 3->5 {100 * x['joint_steps_first_gap_rel']:.3f}% and 3->5->7 "
          f"{100 * x['joint_steps_gap_rel']:.3f}%, joint last two {100 * x['joint_last2_gap_rel']:.3f}%, "
          f"optimal extension of the base {'yes' if x['extension_property_b4'] else 'no'}")
ck(f"order study: {len(fresh)} recomputed orders agree with order_study.json", not bad, f"{bad[:3]}" if bad else "")
if RUN_FULL:
    json.dump(fresh_tasks, open(os.path.join(OUT, "results", "order_study.json"), "w"), default=float)
    STUDY = fresh_tasks

7 orders in 7.4 min
alpha = 0.5  base BTN-MMR-AFG, then LKA-MDV-BGD-NPL: sequential 0.608%, joint 3->5 2.825% and 3->5->7 2.641%, joint last two 0.400%, optimal extension of the base yes
alpha = 0.5  base MDV-LKA-AFG, then BTN-MMR-BGD-NPL: sequential 0.255%, joint 3->5 2.703% and 3->5->7 5.975%, joint last two 0.400%, optimal extension of the base yes
alpha = 0.75 base MMR-LKA-AFG, then NPL-BTN-MDV-BGD: sequential 0.078%, joint 3->5 1.269% and 3->5->7 0.000%, joint last two 0.078%, optimal extension of the base yes
alpha = 0.75 base BTN-MDV-LKA, then NPL-MMR-BGD-AFG: sequential 0.078%, joint 3->5 1.361% and 3->5->7 3.352%, joint last two 0.078%, optimal extension of the base yes
alpha = 0.85 base BTN-MDV-MMR, then NPL-LKA-BGD-AFG: sequential 0.063%, joint 3->5 0.000% and 3->5->7 0.063%, joint last two 0.063%, optimal extension of the base yes
alpha = 0.85 base MDV-LKA-AFG, then BGD-MMR-BTN-NPL: sequential 0.000%, joint 3->5 1.958% and 3->5->7 1.702%, joint last two 0.000%, optimal exte

Table 3 counts, for each exponent, the orders in which the sequential induction and the joint insertion in two-target steps without window miss the exact seven-target tree, with the largest excess, and the orders in which no extension of an optimal base topology is optimal for the seven targets, so that no insertion without window can be exact. It is rebuilt from the stored study (from the fresh one with `RUN_FULL`) and compared with `overleaf_A/tables/tab_vaccine_order.tex`.

In [14]:
summ = RT.vaccine_order_summary(STUDY)
text = RT.table_vaccine_order(summ, os.path.join(OUT, "tables", "tab_vaccine_order.tex"))
ok, msg = RT.compare_table(text, os.path.join(TABLES, "tab_vaccine_order.tex"))
ck("Table 3 identical to overleaf_A/tables/tab_vaccine_order.tex", ok, msg)
print(text)
print(pd.DataFrame(summ).T.to_string())
S = {a: summ[a] for a in RT.ALPHAS}
ck("Table 3: sequential misses 446, 420, 210 of 840, by up to 0.608, 0.078, 0.063%",
   [S[a]["seq_miss"] for a in RT.ALPHAS] == [446, 420, 210] and [round(100 * S[a]["seq_max"], 3) for a in RT.ALPHAS] == [0.608, 0.078, 0.063])
ck("Table 3: joint misses 432, 380, 396, by up to 5.975, 3.352, 1.702%; no optimal extension in 96, 48, 24 orders",
   [S[a]["joint_miss"] for a in RT.ALPHAS] == [432, 380, 396] and [round(100 * S[a]["joint_max"], 3) for a in RT.ALPHAS] == [5.975, 3.352, 1.702]
   and [S[a]["no_ext_b4"] for a in RT.ALPHAS] == [96, 48, 24])
ck("joint insertion of the last two targets from the exact five-target tree: 360, 320, 280 misses, up to 2.241, 0.474, 0.148%",
   [S[a]["joint_last2_miss"] for a in RT.ALPHAS] == [360, 320, 280]
   and [round(100 * S[a]["joint_last2_max"], 3) for a in RT.ALPHAS] == [2.241, 0.474, 0.148])
rows_now = [x for r in STUDY for x in r["rows"]]
misses = [x[k] for x in rows_now for k in ("sequential_gap_rel", "joint_steps_gap_rel", "joint_last2_gap_rel") if x[k] > RT.TIE]
only_joint = [sum(x["joint_steps_gap_rel"] > RT.TIE and x["sequential_gap_rel"] <= RT.TIE for x in rows_now if x["alpha"] == a)
              for a in RT.ALPHAS]
ck("every miss is above 1e-5 relative; the joint insertion misses where the sequential induction is exact in 93, 96, 218 orders",
   min(misses) > 1e-5 and only_joint == [93, 96, 218])
exact_by_order = {}                              # (base, order) -> {alpha: sequential induction exact}
for x in rows_now:
    exact_by_order.setdefault((tuple(x["base"]), tuple(x["order"])), {})[x["alpha"]] = x["sequential_gap_rel"] <= RT.TIE
n_every = sum(all(v.values()) for v in exact_by_order.values())
print(f"orders in which the sequential induction is exact at every exponent: {n_every} of {len(exact_by_order)}")
ck("the sequential induction is exact at every exponent in 394 of the 840 orders (Section 5.3), "
   "and every order in which it is exact at 0.5 is exact at 0.75 and 0.85",
   len(exact_by_order) == 840 and n_every == 394
   and all(all(v.values()) for v in exact_by_order.values() if v[0.5]))
worst = max((x for x in rows_now if x["alpha"] == 0.5), key=lambda x: x["joint_steps_gap_rel"])
ck("worst joint insertion at 0.5: base MDV, LKA, AFG, 2.70% after 3 -> 5 and 5.97% at seven",
   worst["base"] == ["MDV", "LKA", "AFG"] and round(100 * worst["joint_steps_first_gap_rel"], 2) == 2.70
   and round(100 * worst["joint_steps_gap_rel"], 2) == 5.97)

PASS  Table 3 identical to overleaf_A/tables/tab_vaccine_order.tex  [identical]
\begin{tabular}{rrrrrr}
\toprule
 & \multicolumn{2}{c}{Sequential} & \multicolumn{2}{c}{Joint} & \\
\cmidrule(lr){2-3}\cmidrule(lr){4-5}
$\alpha$ & Misses & Largest (\%) & Misses & Largest (\%) & \begin{tabular}[b]{@{}c@{}}No optimal\\extension\end{tabular} \\
\midrule
0.5 & 446/840 & 0.608 & 432/840 & 5.975 & 96/840 \\
0.75 & 420/840 & 0.078 & 380/840 & 3.352 & 48/840 \\
0.85 & 210/840 & 0.063 & 396/840 & 1.702 & 24/840 \\
\bottomrule
\end{tabular}

      orders  seq_miss   seq_max  joint_miss  joint_max  joint_last2_miss  joint_last2_max  no_ext_b4  no_ext_b2_from5  seq_miss_with_ext_b4  joint_miss_with_ext_b4  base_sets
0.50   840.0     446.0  0.006076       432.0   0.059746             360.0         0.022414       96.0            280.0                 350.0                   336.0       35.0
0.75   840.0     420.0  0.000782       380.0   0.033517             320.0         0.004744       48.0            

The joint insertion misses more often, and by more, than the sequential induction. Two targets inserted on the same edge receive separate branching points (Definition 3.8), so they cannot share a sub-branch of their own; the sequential induction reaches such a pair by inserting the second target on an edge created by the first. This is the restriction of the reach in Proposition 3.9, and the joint program, which searches $R_2(T)$ and not $\operatorname{Ext}_2(T)$, inherits it.

## 7. The extension step on every rung

A rung is a set $S$ of three to six targets and one more target $t$, 294 rungs per exponent. On each rung `RT.vaccine_rung` (`vaccine_pune.rung_task`) extends every optimal topology of $S$ by $t$ on every edge and compares the best extension with $d(S\cup\lbrace t\rbrace)$; it also checks directly whether some optimal topology of $S\cup\lbrace t\rbrace$ restricts to an optimal topology of $S$, and evaluates the certificate of Theorem 3.3 with $b=1$ and the polished gap,

$$\Delta_1-\sigma_1<g(S),\qquad \sigma_1=|O-y_t|\Bigl(m(S\cup\lbrace t\rbrace)^{\alpha}-m(S)^{\alpha}\Bigr).$$

The default run recomputes the 420 rungs from three targets.

In [15]:
RUNGS = load("rung_study.json")
names_idx = {t: i for i, t in enumerate(NAMES)}
tasks = [(r["alpha"], tuple(names_idx[t] for t in r["S"]), names_idx[r["t"]]) for r in RUNGS if RUN_FULL or r["n"] == 3]
t0 = time.perf_counter()
fresh = RT.pmap(RT.vaccine_rung, tasks)
print(f"{len(fresh)} rungs in {time.perf_counter() - t0:.0f} s")
by_key = {(r["alpha"], tuple(r["S"]), r["t"]): r for r in RUNGS}
bad = [(r["alpha"], r["S"], r["t"]) for r in fresh if not RT.agrees(r, by_key[(r["alpha"], tuple(r["S"]), r["t"])],
                                                                    rel=1e-7, atol=1e-9)[0]]
ck(f"rungs: {len(fresh)} recomputed rungs agree with rung_study.json", not bad, f"{bad[:3]}" if bad else "")
if RUN_FULL:
    json.dump(fresh, open(os.path.join(OUT, "results", "rung_study.json"), "w"), indent=1, default=float)
    RUNGS = fresh

for a in RT.ALPHAS:
    ra = [r for r in RUNGS if r["alpha"] == a]
    fail = [r for r in ra if not r["extension_property"]]
    by_n = [f"{sum(r['n'] == n for r in fail)}/{sum(r['n'] == n for r in ra)}" for n in (3, 4, 5, 6)]
    targets = pd.Series([r["t"] for r in fail]).value_counts().to_dict()
    print(f"alpha = {a}: no restriction on {len(fail)} rungs ({', '.join(by_n)} for 3->4 ... 6->7), largest excess "
          f"{100 * max(r['extension_excess_rel'] for r in ra):.3f}%, added targets {targets}; certificate on "
          f"{sum(r['certified'] and np.isfinite(r['g_rel']) for r in ra)} rungs with a finite gap and "
          f"{sum(r['certified'] and not np.isfinite(r['g_rel']) for r in ra)} with an infinite one")
count = lambda a, n: sum(not r["extension_property"] for r in RUNGS if r["alpha"] == a and r["n"] == n)
ck("no restriction on 7, 13, 9, 2 rungs at 0.5 (31), 2, 6, 6, 2 at 0.75 (16), 1, 3, 3, 1 at 0.85 (8)",
   [[count(a, n) for n in (3, 4, 5, 6)] for a in RT.ALPHAS] == [[7, 13, 9, 2], [2, 6, 6, 2], [1, 3, 3, 1]])
ck("largest excess of the extension step 0.849, 0.096, 0.074%; failing added targets NPL 14, BGD 11, MMR 4, MDV 2 at 0.5",
   [round(100 * max(r["extension_excess_rel"] for r in RUNGS if r["alpha"] == a), 3) for a in RT.ALPHAS] == [0.849, 0.096, 0.074]
   and pd.Series([r["t"] for r in RUNGS if r["alpha"] == 0.5 and not r["extension_property"]]).value_counts().to_dict()
   == {"NPL": 14, "BGD": 11, "MMR": 4, "MDV": 2})
ck("restriction and exactness of the extension step agree on all 882 rungs",
   len(RUNGS) == 882 and all(r["extension_property"] == r["extension_exact"] for r in RUNGS))
ck("the certificate holds on 14, 19, 21 rungs with a finite gap and 0, 20, 32 with an infinite one, never on a failing rung",
   [sum(r["certified"] and np.isfinite(r["g_rel"]) for r in RUNGS if r["alpha"] == a) for a in RT.ALPHAS] == [14, 19, 21]
   and [sum(r["certified"] and not np.isfinite(r["g_rel"]) for r in RUNGS if r["alpha"] == a) for a in RT.ALPHAS] == [0, 20, 32]
   and not any(r["certified"] and not r["extension_exact"] for r in RUNGS))
flags = {}
for x in [x for r in STUDY for x in r["rows"]]:
    perm = x["base"] + x["order"]
    for n in (3, 4, 5, 6):
        flags.setdefault((x["alpha"], tuple(sorted(perm[:n], key=NAMES.index)), perm[n]), set()).add(
            x["extension_property_rungs"][f"{n}->{n + 1}"])
ck("the rung flags of the order study match the rung study (0 mismatches)",
   all(flags.get((r["alpha"], tuple(r["S"]), r["t"])) == {r["extension_property"]} for r in RUNGS))

420 rungs in 660 s
PASS  rungs: 420 recomputed rungs agree with rung_study.json
alpha = 0.5: no restriction on 31 rungs (7/140, 13/105, 9/42, 2/7 for 3->4 ... 6->7), largest excess 0.849%, added targets {'NPL': 14, 'BGD': 11, 'MMR': 4, 'MDV': 2}; certificate on 14 rungs with a finite gap and 0 with an infinite one
alpha = 0.75: no restriction on 16 rungs (2/140, 6/105, 6/42, 2/7 for 3->4 ... 6->7), largest excess 0.096%, added targets {'NPL': 8, 'BGD': 8}; certificate on 19 rungs with a finite gap and 20 with an infinite one
alpha = 0.85: no restriction on 8 rungs (1/140, 3/105, 3/42, 1/7 for 3->4 ... 6->7), largest excess 0.074%, added targets {'BGD': 8}; certificate on 21 rungs with a finite gap and 32 with an infinite one
PASS  no restriction on 7, 13, 9, 2 rungs at 0.5 (31), 2, 6, 6, 2 at 0.75 (16), 1, 3, 3, 1 at 0.85 (8)
PASS  largest excess of the extension step 0.849, 0.096, 0.074%; failing added targets NPL 14, BGD 11, MMR 4, MDV 2 at 0.5
PASS  restriction and exactness of the 

## 8. The gap certificate with rigorous lower bounds and Table 5

**Theorem 3.3** with $b=1$: if

$$\Delta_1(T)-\sigma_1<g_n,$$

then every optimal topology of $S_{n+1}$ restricts to an optimal topology of $S_n$, and the extension step is exact. A computed $g_n$ comes from polished costs, which are upper bounds on the minima of their topologies. **Proposition 3.4** replaces them by lower bounds: with $\widehat{\mathcal T}_n$ the topologies within $10^{-7}$ of $d_n$, $\ell(T'')\le c_n(T'')$ for every other topology, and $c_{\mathrm{ext}}$ the cost of the best extension,

$$c_{\mathrm{ext}}-\sigma_1<\min_{T''\notin\widehat{\mathcal T}_n}\ell(T'')$$

certifies the restriction. The bound $\ell$ is weak duality for the geometry problem of a fixed topology (`RT.dual_lower_bound`), and the gap from lower bounds is

$$g_n^{\mathrm{rig}}=\min_{T''\notin\widehat{\mathcal T}_n}\ell(T'')-d_n .$$

**Corollary 3.6** bounds the next gap without enumerating $S_{n+1}$: with $\mathcal U$ the union of the extensions of the optimal topologies of $S_n$,

$$g_{n+1}\ge\min\Bigl\lbrace\min_{T'\in\mathcal U\setminus\mathcal T^{\ast}_{n+1}}c_{n+1}(T'),\ d_n+g_n+\sigma_1\Bigr\rbrace-d_{n+1}.$$

The rows of the vaccine network in Table 5 are the four rungs of the date order at each exponent, with $g_n^{\mathrm{rig}}$ in the gap column and the certificate decided against it. `RT.vaccine_cert_task` runs `RT.certificate_rung` (`inductive_certificate.rung`) and `RT.rigorous_rung` (`certificate_rigorous.one`); the default run recomputes the rungs $3\to4$ and $4\to5$.

In [16]:
CV = load("certificate_vaccine.json")
nmax = 6 if RUN_FULL else 4
tasks = [(kind, a, n) for kind in ("rung", "rigorous") for a in RT.ALPHAS for n in range(3, nmax + 1)]
t0 = time.perf_counter()
fresh = RT.pmap(RT.vaccine_cert_task, tasks)
print(f"{len(fresh)} certificate computations in {time.perf_counter() - t0:.0f} s")
bad = []
for kind, r in fresh:
    old = [x for x in CV[kind] if x["alpha"] == r["alpha"] and x["n"] == r["n"]][0]
    ok, msg = RT.agrees(r, old, rel=1e-7, atol=1e-9)
    if not ok:
        bad.append(f"{kind} {r['alpha']} {r['n']}: {msg}")
ck(f"certificate: {len(fresh)} recomputed rungs agree with certificate_vaccine.json", not bad, "; ".join(bad[:2]))
rows = RT.vaccine_certificate_rows(CV["rung"], CV["rigorous"])
ok, msg = RT.agrees(rows, CV["rows"], rel=1e-12, atol=1e-12)
ck("the combined rows of certificate_vaccine.json follow from its rung and rigorous parts", ok, msg)
if RUN_FULL:
    rows = RT.vaccine_certificate_rows([r for k, r in fresh if k == "rung"], [r for k, r in fresh if k == "rigorous"])

12 certificate computations in 78 s
PASS  certificate: 12 recomputed rungs agree with certificate_vaccine.json
PASS  the combined rows of certificate_vaccine.json follow from its rung and rigorous parts


In [17]:
DET = RT.deterministic_instances()
LADDER = json.load(open(os.path.join(FIG, "inductive", "inductive_ladder.json")))
HARD = json.load(open(os.path.join(FIG, "inductive", "inductive_ladder_hard.json")))
CERT = [r for r in json.load(open(os.path.join(FIG, "inductive", "certificate_ladder.json"))) if r["instance"] in DET]
text = RT.table_certificate_article(rows, CERT, LADDER, HARD, os.path.join(OUT, "tables", "tab_certificate.tex"))
ok, msg = RT.compare_table(text, os.path.join(TABLES, "tab_certificate.tex"))
ck("Table 5 identical to overleaf_A/tables/tab_certificate.tex (vaccine rows first)", ok, msg)
print("\n".join(text.splitlines()[:16]), "\n...")
print(pd.DataFrame([{"alpha": r["alpha"], "rung": f"{r['n']}->{r['n_next']}", "g_n (%)": r["gap_rigorous_rel_pct"],
                     "Delta_1 (%)": r["increment_rel_pct"], "sigma_1 (%)": r["saving_rel_pct"], "net (%)": r["net_rel_pct"],
                     "certified": r["certified_rigorous"], "margin / d_n": r["rigorous_margin_rel"],
                     "Cor. 3.6 bound (%)": 100 * r["cor_bound_rel"], "g_(n+1) (%)": 100 * r["g_next_rel"]} for r in rows]).to_string(index=False))
c0 = [r for r in rows if r["certified_rigorous"]]
ck("the certificate holds only at alpha = 0.5 on the rung 3 -> 4, margin 9.3e-5 d_n (9.350 against 9.34 net)",
   len(c0) == 1 and (c0[0]["alpha"], c0[0]["n"]) == (0.5, 3) and round(c0[0]["rigorous_margin_rel"], 6) == 9.3e-5
   and round(c0[0]["gap_rigorous_rel_pct"], 3) == 9.350 and round(c0[0]["net_rel_pct"], 2) == 9.34)
ck("Corollary 3.6 holds on all 12 rows; 0.0079 against g_4 = 0.3138% on the certified rung, negative bound elsewhere",
   len(rows) == 12 and all(r["cor_holds"] for r in rows) and round(100 * c0[0]["cor_bound_rel"], 4) == 0.0079
   and round(100 * c0[0]["g_next_rel"], 4) == 0.3138 and all(r["cor_bound_rel"] < 0 for r in rows if r is not c0[0]))
ck("one optimal topology and an exact extension step on every row; rigorous and polished gaps agree to 1e-11, dual slack 3e-12",
   all(r["n_optimal"] == 1 and r["extension_exact"] for r in rows)
   and max(abs(r["gap_rigorous_rel_pct"] - r["gap_rel_pct"]) / 100 for r in rows) < 1e-11
   and round(max(r["dual_slack_rel_max"] for r in rows), 12) == 3e-12)

PASS  Table 5 identical to overleaf_A/tables/tab_certificate.tex (vaccine rows first)  [identical]
\begin{tabular}{lrlrrrrrll}
\toprule
Instance & $\alpha$ & Rung & $|\mathcal T^{*}_{n}|$ & $g_{n}$ (\%) & $\Delta_{1}$ (\%) & $\sigma_{1}$ (\%) & \begin{tabular}[b]{@{}c@{}}$\Delta_{1}-\sigma_{1}$\\(\%)\end{tabular} & Certified & \begin{tabular}[b]{@{}c@{}}Extension\\exact\end{tabular} \\
\midrule
Vaccine network & 0.5 & $3\to4$ & 1 & 9.350 & 16.95 & 7.61 & 9.34 & yes & yes \\
Vaccine network & 0.5 & $4\to5$ & 1 & 0.314 & 32.40 & 12.72 & 19.68 & no & yes \\
Vaccine network & 0.5 & $5\to6$ & 1 & 0.202 & 11.70 & 1.95 & 9.75 & no & yes \\
Vaccine network & 0.5 & $6\to7$ & 1 & 0.181 & 14.28 & 1.67 & 12.60 & no & yes \\
Vaccine network & 0.75 & $3\to4$ & 1 & 4.366 & 19.38 & 13.06 & 6.32 & no & yes \\
Vaccine network & 0.75 & $4\to5$ & 1 & 0.204 & 35.73 & 22.57 & 13.16 & no & yes \\
Vaccine network & 0.75 & $5\to6$ & 1 & 0.006 & 8.46 & 3.50 & 4.96 & no & yes \\
Vaccine network & 0.75 & $6\to7$ 

## 9. Every seven-target topology and the rigorous gap $g_7$

Above 1000 topologies the exact solver polishes only the topologies within 25 percent of the best scan cost, at most 5000, so the threshold map needs a gap of the seven-target instance that does not depend on that selection. Every one of the $(2\cdot8-5)!!=10\,395$ full topologies on eight terminals is therefore scanned and polished at each exponent, its cost taken as the smaller of the two, and given the larger of the weak-duality bounds built from the scan and from the polished positions (`RT.vaccine_fullpolish_chunk`, `vaccine_pune.fullpolish_chunk`). Then

$$g_7^{\mathrm{rig}}=\min_{T\notin\mathcal T^{\ast}_7}\ell(T)-d_7 .$$

The default run recomputes the first 210 topologies at each exponent, the optimal ones, and the topology with the least lower bound outside them; the summary is rebuilt from the 31 185 stored rows. The population variant of Section 11 was polished in the same way, without lower bounds (`python vaccine_pune.py fullpolish_population`), to check its capped exact solve; the default run recomputes its first 35 topologies at each exponent.

In [18]:
FP = load("full_polish_7.json")
N7 = RC.count_full_topologies(8)
tasks = []
for a in RT.ALPHAS:
    if RUN_FULL:
        tasks += [("doses", a, st, min(st + 105, N7), True) for st in range(0, N7, 105)]
        tasks += [("population", a, st, min(st + 105, N7), False) for st in range(0, N7, 105)]
        continue
    tasks.append(("population", a, 0, 35, False))     # the population variant: first 35 topologies, no lower bound
    rows_a = FP["rows"][f"doses|{a}"]
    ref7 = EXACT[RT.vaccine_key("doses", a, tuple(range(7)))]
    ties = {frozenset(RT.split_of(t["edges"], 8)) for t in ref7["ties"]}
    special = [i for i, r in enumerate(rows_a) if frozenset(r["splits"]) in ties]
    outside = [i for i in range(N7) if i not in special]
    special.append(min(outside, key=lambda i: rows_a[i]["lower_bound"]))
    tasks += [("doses", a, st, st + 35, True) for st in range(0, 210, 35)] + [("doses", a, i, i + 1, True) for i in special]
t0 = time.perf_counter()
fresh = RT.pmap(RT.vaccine_fullpolish_chunk, tasks)
print(f"{sum(len(o) for *_, o in fresh)} topologies polished in {time.perf_counter() - t0:.0f} s")
bad = []
for variant, a, st, out in fresh:
    ok, msg = RT.agrees(out, FP["rows"][f"{variant}|{a}"][st:st + len(out)], rel=1e-7, atol=1e-9)
    if not ok:
        bad.append(f"{a} from {st}: {msg}")
ck(f"full polish: {sum(len(o) for *_, o in fresh)} recomputed topologies agree with full_polish_7.json", not bad, "; ".join(bad[:2]))
if RUN_FULL:
    for v in ("doses", "population"):
        for a in RT.ALPHAS:
            FP["rows"][f"{v}|{a}"] = [r for vv, aa, st, out in sorted(fresh, key=lambda t: (t[1], t[2]))
                                      if vv == v and aa == a for r in out]

GAP7 = {}
for a in RT.ALPHAS:
    summ = RT.vaccine_fullpolish_summary(FP["rows"][f"doses|{a}"], EXACT[RT.vaccine_key("doses", a, tuple(range(7)))])
    ok, msg = RT.agrees(summ, FP["summary"][f"doses|{a}"], rel=1e-9, atol=1e-12)
    ck(f"full-polish summary at alpha = {a} as stored", ok, msg)
    GAP7[a] = summ
    print(f"alpha = {a}: d_7 against the full polish {summ['rel_diff_full_polish']:+.1e}; g_7 polished "
          f"{100 * summ['g_full_polish_rel']:.5f}%, from the solver {100 * summ['g_reference_rel']:.5f}%, rigorous "
          f"{100 * summ['g_rigorous_rel']:.5f}%; largest dual slack {summ['dual_slack_rel_max_outside_S']:.1e}")
ck("the full polish reproduces d_7 to 2e-16 and g_7 to 1e-15; the rigorous g_7 is within 1e-12 of it",
   all(abs(GAP7[a]["rel_diff_full_polish"]) <= 2.3e-16 and abs(GAP7[a]["g_full_polish_rel"] - GAP7[a]["g_reference_rel"]) < 1e-15
       and 0 <= GAP7[a]["g_full_polish_rel"] - GAP7[a]["g_rigorous_rel"] < 1e-12 for a in RT.ALPHAS))

# The population variant: the full polish against its capped exact solve (vaccine_pune.step_fullpolish_summary)
POP7 = {}
for a in RT.ALPHAS:
    ref_p = EXACT[RT.vaccine_key("population", a, tuple(range(7)))]
    ties_p = {frozenset(RT.split_of(t["edges"], 8)) for t in ref_p["ties"]}
    rows_p = FP["rows"][f"population|{a}"]
    pol = np.array([r.get("cost", r["polished"]) for r in rows_p])
    in_S = np.array([frozenset(r["splits"]) in ties_p for r in rows_p])
    d_p = ref_p["d"]
    POP7[a] = {"topologies": len(rows_p), "d_reference": d_p, "min_full_polish": float(pol.min()),
               "rel_diff_full_polish": float(pol.min() / d_p - 1), "tie_topologies": int(in_S.sum()),
               "ties_reference": len(ref_p["ties"]), "g_full_polish_rel": float((pol[~in_S].min() - d_p) / d_p),
               "g_reference_rel": ref_p["g_rel"], "outside_S_within_1e-7": int(np.sum(pol[~in_S] <= d_p * (1 + RT.TIE)))}
    print(f"population, alpha = {a}: capped solve polished {ref_p['n_polished']} topologies; d_7 against the full polish "
          f"{POP7[a]['rel_diff_full_polish']:+.1e}; g_7 {100 * POP7[a]['g_full_polish_rel']:.5f}% "
          f"(capped solve {100 * ref_p['g_rel']:.5f}%)")
ck("the cap of 5000 polished topologies decides all six eight-terminal solves (dose and population masses), and the "
   "full polish of all 10 395 topologies reproduces them to 3e-14 relative (Section 4.1), as stored",
   all(RT.agrees(POP7[a], FP["summary"][f"population|{a}"], rel=1e-9, atol=1e-12)[0] for a in RT.ALPHAS)
   and max(abs(r["rel_diff_full_polish"]) for r in list(POP7.values()) + list(GAP7.values())) < 3e-14
   and all(abs(POP7[a]["g_full_polish_rel"] - POP7[a]["g_reference_rel"]) < 1e-14 for a in RT.ALPHAS)
   and all(EXACT[RT.vaccine_key(v, a, tuple(range(7)))]["n_polished"] == 5000
           for v in ("doses", "population") for a in RT.ALPHAS))

745 topologies polished in 182 s
PASS  full polish: 745 recomputed topologies agree with full_polish_7.json
PASS  full-polish summary at alpha = 0.5 as stored
alpha = 0.5: d_7 against the full polish +2.2e-16; g_7 polished 0.21048%, from the solver 0.21048%, rigorous 0.21048%; largest dual slack 1.9e-11
PASS  full-polish summary at alpha = 0.75 as stored
alpha = 0.75: d_7 against the full polish +2.2e-16; g_7 polished 0.00520%, from the solver 0.00520%, rigorous 0.00520%; largest dual slack 1.2e-11
PASS  full-polish summary at alpha = 0.85 as stored
alpha = 0.85: d_7 against the full polish +0.0e+00; g_7 polished 0.05704%, from the solver 0.05704%, rigorous 0.05704%; largest dual slack 1.8e-11
PASS  the full polish reproduces d_7 to 2e-16 and g_7 to 1e-15; the rigorous g_7 is within 1e-12 of it
population, alpha = 0.5: capped solve polished 5000 topologies; d_7 against the full polish +0.0e+00; g_7 0.03772% (capped solve 0.03772%)
population, alpha = 0.75: capped solve polished 5000 to

## 10. The threshold map (Corollary 3.7)

**Corollary 3.7.** Let $(T,P)$ be an optimal realization for $S_n$ and let a new recipient $y$ have demand $\varepsilon$. For a point $q$ of the tree let $w_e$ be the flows on the path from $O$ to $q$, the edge that contains $q$ counted up to $q$. If for some $q$

$$\sum_{e\in\operatorname{path}(O,q)}|e|\bigl((w_e+\varepsilon)^{\alpha}-w_e^{\alpha}\bigr)+\varepsilon^{\alpha}|y-q|-|O-y|\Bigl(\bigl(m^{(n)}+\varepsilon\bigr)^{\alpha}-\bigl(m^{(n)}\bigr)^{\alpha}\Bigr)<g_n,$$

then every optimal topology of $S_n\cup\lbrace y\rbrace$ restricts to an optimal topology of $S_n$: the request can be grafted onto the existing network without reorganizing it.

For each $q$ the left side is nondecreasing in $\varepsilon$, so the certified demands form an interval $[0,\varepsilon^{\ast}(y))$. `RT.vaccine_threshold_map` (`vaccine_pune.step_threshold`) samples 120 points $q$ on every edge of the optimal seven-target tree (`RT.tree_samples`), takes the minimum of the left side over them, which is an upper bound on the minimum over the tree, so $\varepsilon^{\ast}$ is conservative, and finds $\varepsilon^{\ast}(y)$ by bisection in $\log\varepsilon$ between $10^{-12}m^{(7)}$ and $10\,m^{(7)}$ on a $140\times140$ grid over the convex hull of the eight sites. The gap is the smallest of $g_7^{\mathrm{rig}}$ and the two polished values of Section 9, which is $g_7^{\mathrm{rig}}$ at every exponent. Thresholds are in thousands of doses.

In [19]:
TM = load("threshold_map.json")
tasks = []
for a in RT.ALPHAS:
    d, g_pol, _ = RT.vaccine_exact_of(tuple(range(7)), a)
    g_rig, g_full = GAP7[a]["g_rigorous_rel"] * d, GAP7[a]["g_full_polish_rel"] * d
    g_use = min(g_rig, g_pol, g_full)
    tasks.append((a, g_use, "rigorous lower bound" if g_use == g_rig else "polished", 140, 0.06))
t0 = time.perf_counter()
maps = RT.pmap(RT.vaccine_threshold_map, tasks)
print(f"three maps of 19 600 points in {time.perf_counter() - t0:.0f} s")
for r in maps:
    ok, msg = RT.agrees(strip(r), strip(TM["maps"][str(r["alpha"])]), rel=1e-6, atol=1e-5)
    ck(f"threshold map at alpha = {r['alpha']} as stored (gap: {r['g_used']})", ok and r["g_used"] == "rigorous lower bound", msg)
xs, ys, Y, inside, hull = RT.vaccine_threshold_grid()
ck("grid and convex hull as stored (8584 grid points inside the hull)",
   inside.astype(int).tolist() == TM["inside_hull"] and hull == TM["hull_vertices"] and int(inside.sum()) == 8584)

s, T, m = RT.vaccine_instance()
X8 = dict(zip(("O",) + NAMES, np.vstack([s, T])))

def seg_dist(p, a, b):
    """Distance from the point p to the segment ab (km)."""
    d = b - a
    t = np.clip(np.dot(p - a, d) / max(np.dot(d, d), 1e-12), 0, 1)
    return float(np.linalg.norm(p - (a + t * d)))

rows = []
for r in maps:
    E = np.array(r["eps_kdoses"])[inside]
    xy = np.array(r["argmin_hull_xy"])
    P = np.array(r["tree"]["P"])
    rows.append({"alpha": r["alpha"], "min": E.min(), "median": np.median(E), "10th pct": np.percentile(E, 10),
                 "90th pct": np.percentile(E, 90), "max": E.max(), "argmin (km)": np.round(xy, 1).tolist(),
                 "km to the tree": min(seg_dist(xy, P[u], P[v]) for u, v in r["tree"]["edges"]),
                 "km to the side LKA-MMR": seg_dist(xy, X8["LKA"], X8["MMR"])})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("hull of the sites:", [(("O",) + NAMES)[i] for i in hull])
print("current recipients (thousand doses):", TM["masses_thousand_doses"])
st_ = {r["alpha"]: r for r in rows}
ck("thresholds: minimum 0.810, 0.113, 7.25 and median 14.43, 0.765, 31.09 thousand doses",
   [round(st_[a]["min"], 3) for a in (0.5, 0.75)] == [0.810, 0.113] and round(st_[0.85]["min"], 2) == 7.25
   and [round(st_[a]["median"], 2) for a in (0.5, 0.85)] == [14.43, 31.09] and round(st_[0.75]["median"], 3) == 0.765)
ck("10th to 90th percentile: 2.0-128, 0.21-3.7, 12.3-114 thousand doses",
   (round(st_[0.5]["10th pct"], 1), round(st_[0.5]["90th pct"])) == (2.0, 128)
   and (round(st_[0.75]["10th pct"], 2), round(st_[0.75]["90th pct"], 1)) == (0.21, 3.7)
   and (round(st_[0.85]["10th pct"], 1), round(st_[0.85]["90th pct"])) == (12.3, 114))
top = 1e4 * TM["total_million_doses"]               # the upper end of the bisection, thousand doses
ck("at alpha = 0.5 one grid point reaches the upper end of the bisection, ten times the total (192.4 million doses)",
   int((np.array(maps[0]["eps_kdoses"])[inside] >= top * (1 - 1e-9)).sum()) == 1 and round(top / 1000, 1) == 192.4
   and all(np.array(r["eps_kdoses"])[inside].max() < top * (1 - 1e-9) for r in maps[1:]))
ck("the minimum lies within 300 km of the side LKA-MMR of the hull and more than 900 km from the tree",
   all(r["km to the side LKA-MMR"] < 300 and r["km to the tree"] > 900 for r in rows))

# The grid points on land, where a new recipient would be (vaccine_pune.land_mask and land_statistics)
land, owner = RT.vaccine_land_mask(Y)
on_land, owner = land.reshape(inside.shape), owner.reshape(inside.shape)
smallest = min(TM["masses_thousand_doses"].values())          # the Maldives, 312 thousand doses
LAND = {r["alpha"]: RT.vaccine_land_statistics(np.array(r["eps_kdoses"]), inside, on_land, owner, Y, smallest)
        for r in maps}
print(pd.DataFrame(LAND).T.to_string())
ck("grid points on land as stored: 5509 of the 8584 points of the hull, 3075 at sea",
   on_land.astype(int).tolist() == TM["on_land"] and LAND[0.5]["land_points_hull"] == 5509
   and LAND[0.5]["sea_points_hull"] == 3075)
himalaya = RT.project(31.5, 79.0)                             # 31.5 N, 79.0 E
ck("on land: medians 25.4, 1.12, 39.5 and minima 1.78, 0.16, 8.87 thousand doses, within 100 km of 31.5 N, 79.0 E "
   "(western Himalaya), as stored",
   all(RT.agrees(LAND[a], TM["maps"][str(a)]["land"], rel=1e-6, atol=1e-5)[0] for a in RT.ALPHAS)
   and [round(LAND[a]["eps_median_land_kdoses"], 1) for a in (0.5, 0.85)] == [25.4, 39.5]
   and round(LAND[0.75]["eps_median_land_kdoses"], 2) == 1.12
   and [round(LAND[a]["eps_min_land_kdoses"], 2) for a in RT.ALPHAS] == [1.78, 0.16, 8.87]
   and all(np.linalg.norm(np.array(LAND[a]["argmin_land_xy"]) - himalaya) < 100 for a in RT.ALPHAS))
ck("on land the threshold is below the smallest recipient (312 thousand doses) at 93.2, 99.98, and 95.8% of the points",
   [round(100 * LAND[a]["share_land_below_smallest_recipient"], 2) for a in RT.ALPHAS] == [93.17, 99.98, 95.77])

three maps of 19 600 points in 101 s
PASS  threshold map at alpha = 0.5 as stored (gap: rigorous lower bound)
PASS  threshold map at alpha = 0.75 as stored (gap: rigorous lower bound)
PASS  threshold map at alpha = 0.85 as stored (gap: rigorous lower bound)
PASS  grid and convex hull as stored (8584 grid points inside the hull)
 alpha    min  median  10th pct  90th pct       max     argmin (km)  km to the tree  km to the side LKA-MMR
   0.5 0.8104   14.43     2.019     127.9 1.924e+05 [370.8, -579.0]            1031                   278.7
  0.75 0.1126  0.7647    0.2099     3.681     436.3 [526.3, -771.8]            1016                   32.33
  0.85  7.251   31.09     12.31     113.6 1.243e+04 [526.3, -799.3]           933.4                   9.243
hull of the sites: ['AFG', 'MDV', 'LKA', 'MMR', 'BTN']
current recipients (thousand doses): {'BTN': 550.0, 'MDV': 312.0, 'BGD': 10200.0, 'NPL': 2448.0, 'MMR': 3500.0, 'LKA': 1264.0, 'AFG': 968.0}
PASS  thresholds: minimum 0.810, 0.113, 7.

Every current recipient asked for far more than the threshold anywhere in the hull: Maldives, the smallest, received 312 thousand doses, while the median threshold is 14 thousand doses at $\alpha=0.5$, 0.8 thousand at $0.75$, and 31 thousand at $0.85$. The certificate protects the topology of the network only against new requests of the size of a small consignment. At $\alpha=0.75$ the seven-target gap is only 0.0052 percent of the cost, which is why the thresholds there are the smallest. The minimum lies over the Bay of Bengal, near the side of the hull between Colombo and Yangon and about 1000 km from the nearest edge of the tree, where the pendant term $\varepsilon^{\alpha}|y-q|$ of the corollary is largest. A new recipient would be on land, and 3 075 of the 8 584 grid points of the hull are at sea; `RT.vaccine_land_mask` keeps the points inside a country of `data/maps/countries.geojson`. On land the thresholds are higher, with medians of 25.4, 1.12, and 39.5 thousand doses, and the minimum, 1.78, 0.16, and 8.87 thousand doses, lies in the western Himalaya, near the northern edge of the hull; the threshold is still below the smallest current recipient at more than 93 percent of the land points.

## 11. The population variant

A future pandemic with allocation by population would send each country a share of the supply proportional to its population. The variant keeps the sites and the total of 19.242 million doses and sets

$$m_i^{\mathrm{pop}}=19.242\,\frac{\mathrm{pop}_i}{\sum_j\mathrm{pop}_j}$$

with the World Bank populations of 2021. Its exact seven-target trees are stored with those of Section 3 (about 8 minutes each on one core; recomputed with `RUN_FULL`). `RT.vaccine_population_cross` (`vaccine_pune.pop_task`) re-optimizes the geometry of the optimal topologies of one mass vector under the other, which measures what a network designed for one demand costs under the other.

In [20]:
PV = load("population_variant.json")
mp = RT.vaccine_instance("population")[2]
print("population masses (million doses):", {t: round(float(v), 4) for t, v in zip(NAMES, mp)})
ck("population masses BTN 0.0475, MDV 0.0316, BGD 10.2752, NPL 1.8064, MMR 3.2719, LKA 1.3579, AFG 2.4515",
   [round(float(v), 4) for v in mp] == [0.0475, 0.0316, 10.2752, 1.8064, 3.2719, 1.3579, 2.4515]
   and abs(mp.sum() - 19.242) < 1e-12)
if RUN_FULL:
    for a in RT.ALPHAS:
        r = RT.vaccine_solve_subset(("population", a, tuple(range(7))))
        ok, msg = same_exact(r, EXACT[RT.vaccine_key("population", a, tuple(range(7)))])
        ck(f"population variant, exact seven-target tree at alpha = {a} as stored", ok, msg)
labels = ["O"] + list(NAMES)
for a in RT.ALPHAS:
    rd, rp = (EXACT[RT.vaccine_key(v, a, tuple(range(7)))] for v in ("doses", "population"))
    same = {RT.split_of(t["edges"], 8) for t in rd["ties"]} == {RT.split_of(t["edges"], 8) for t in rp["ties"]}
    print(f"alpha = {a}: d = {rp['d']:.4f}, g = {100 * rp['g_rel']:.4f}%, same optimal topologies as the doses: {same}; "
          f"tree {RT.nested_tree(rp['best']['edges'], rp['best']['P'], labels)}")
    ck(f"population variant at alpha = {a}: same topologies as stored ({same})", same == PV["alphas"][str(a)]["same_optimal_full_topologies"])
pp = {a: EXACT[RT.vaccine_key("population", a, tuple(range(7)))] for a in RT.ALPHAS}
ck("population trees: d = 14159.0233, 22389.2703, 27152.7264; g = 0.0377, 0.0164, 0.0122%; other topology only at 0.75",
   [round(pp[a]["d"], 4) for a in RT.ALPHAS] == [14159.0233, 22389.2703, 27152.7264]
   and [round(100 * pp[a]["g_rel"], 4) for a in RT.ALPHAS] == [0.0377, 0.0164, 0.0122]
   and [PV["alphas"][str(a)]["same_optimal_full_topologies"] for a in RT.ALPHAS] == [True, False, True])
tree = lambda r: RT.nested_tree(r["best"]["edges"], r["best"]["P"], labels)
ck("at 0.75 the population tree is the dose tree of 0.85, O((((BGD,BTN),MMR),NPL),(LKA,MDV),AFG)",
   tree(pp[0.75]) == tree(EXACT[RT.vaccine_key("doses", 0.85, tuple(range(7)))]) == "O((((BGD,BTN),MMR),NPL),(LKA,MDV),AFG)")

tasks = [(a, src, dst) for a in RT.ALPHAS for src, dst in (("doses", "population"), ("population", "doses"))]
cross = RT.pmap(RT.vaccine_population_cross, tasks)
bad = []
for r in cross:
    old = [c for c in PV["alphas"][str(r["alpha"])]["cross"] if c["topology_of"] == r["topology_of"]][0]
    ok, msg = RT.agrees(strip(r), strip(old), rel=1e-7, atol=1e-9)
    if not ok:
        bad.append(msg)
    print(f"alpha = {r['alpha']}: optimal topologies of the {r['topology_of']} under the masses of the {r['masses_of']}: "
          f"{100 * r['excess_rel']:+.4f}% above the optimum")
ck("cross re-optimizations as stored", not bad, "; ".join(bad[:2]))
x75 = {r["topology_of"]: r["excess_rel"] for r in cross if r["alpha"] == 0.75}
ck("at 0.75 the dose topology costs +0.0164% under population masses and the population topology +0.0052% under doses",
   round(100 * x75["doses"], 4) == 0.0164 and round(100 * x75["population"], 4) == 0.0052
   and all(abs(r["excess_rel"]) < 1e-9 for r in cross if r["alpha"] != 0.75))

population masses (million doses): {'BTN': 0.0475, 'MDV': 0.0316, 'BGD': 10.2752, 'NPL': 1.8064, 'MMR': 3.2719, 'LKA': 1.3579, 'AFG': 2.4515}
PASS  population masses BTN 0.0475, MDV 0.0316, BGD 10.2752, NPL 1.8064, MMR 3.2719, LKA 1.3579, AFG 2.4515
alpha = 0.5: d = 14159.0233, g = 0.0377%, same optimal topologies as the doses: True; tree O((((BGD(MMR),BTN),NPL),AFG),(LKA,MDV))
PASS  population variant at alpha = 0.5: same topologies as stored (True)
alpha = 0.75: d = 22389.2703, g = 0.0164%, same optimal topologies as the doses: False; tree O((((BGD,BTN),MMR),NPL),(LKA,MDV),AFG)
PASS  population variant at alpha = 0.75: same topologies as stored (False)
alpha = 0.85: d = 27152.7264, g = 0.0122%, same optimal topologies as the doses: True; tree O((((BGD,BTN),MMR),NPL),(LKA,MDV),AFG)
PASS  population variant at alpha = 0.85: same topologies as stored (True)
PASS  population trees: d = 14159.0233, 22389.2703, 27152.7264; g = 0.0377, 0.0164, 0.0122%; other topology only at 0.75
PASS  at 0

## 12. Maps

The maps of Section 5.3 are drawn by `code/figure_vaccine_maps.py`, the script that made the figures of the paper; this cell imports it and draws them from the stored results into `colab/output/figures`. Every map uses the projection of Section 2 and the Natural Earth layers of `data/maps`; national borders are those of Natural Earth and imply no position on any boundary.

- `fig_vaccine_network`, at $\alpha=0.5$: (a) the exact tree on the first three recipients of the date order, BTN, MDV, and BGD, the later recipients as open circles; (b) the sequential induction to seven targets in the order that inserts Myanmar before Nepal, 0.255 percent above the optimum (Section 4); (c) the exact seven-target tree, which the sequential induction along the date order reaches.
- `fig_vaccine_threshold`: the certified threshold of Section 10 over the convex hull of the sites, one panel per exponent, on a common logarithmic scale.
- `fig_vaccine_population`: the eastern part of the optimal tree at $\alpha=0.75$ with the dose masses and with the population masses (Section 11).

In [21]:
sys.path.append(os.path.join(ROOT, "code"))        # figure_vaccine_maps.py with vaccine_pune.py and plotstyle.py
import matplotlib.pyplot as plt
import figure_vaccine_maps as FM

for name, draw in (("fig_vaccine_network", FM.fig_network), ("fig_vaccine_threshold", FM.fig_threshold),
                   ("fig_vaccine_population", FM.fig_population)):
    fig, axes, rep = draw(save=False)
    paths = [os.path.join(OUT, "figures", f"{name}.{ext}") for ext in ("pdf", "png")]
    for p in paths:
        fig.savefig(p)
    plt.close(fig)
    display(Image(paths[1], width=900))
    stored_png = os.path.join(VFIG, f"{name}.png")
    same = os.path.exists(stored_png) and filecmp.cmp(paths[1], stored_png, shallow=False)
    print(f"{name}: PNG identical to figures/vaccine_pune/{name}.png: {same} "
          "(differences in fonts or library versions, as in Colab, change the bytes but not the content)")

Output hidden; open in https://colab.research.google.com to view.

## 13. Summary

Every check of this notebook, and the time it took. The rebuilt tables are in `colab/output/tables` and the maps in `colab/output/figures`; with `RUN_FULL` the recomputed results are written to `colab/output/results`.

In [22]:
all_passed = ck.summary()
print(f"notebook 3 ran in {(time.perf_counter() - T_START) / 60:.1f} minutes (RUN_FULL = {RUN_FULL})")
print("PASS" if all_passed else "FAIL")

66 of 69 checks passed
  FAILED: processed coordinates (km), the 28 distances, and the three candidate projections equal those computed above (bit for bit)
  FAILED: date order: 12 recomputed chains agree with primary_order.json
  FAILED: cross re-optimizations as stored
notebook 3 ran in 37.6 minutes (RUN_FULL = False)
FAIL
